In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", 'vllm==0.32.0')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", 'pip uninstall -y torchaudio'):
        sh(os.environ.get("PIP_POST", 'pip uninstall -y torchaudio'))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    'llm.py': '"""Greedy chat generation shared by the screening runs: vLLM (LLM_BACKEND=vllm, default) or transformers (LLM_BACKEND=hf).\n\n    m = LLM("Qwen/Qwen3-4B-Instruct-2507", max_len=16384, tp=1)\n    texts = m.gen([[{"role": "system", "content": ...}, {"role": "user", "content": ...}], ...], max_tokens=512)\n\nOne process loads one repo; trying the next candidate (fp16 -> AWQ, vllm -> hf) is multirun.py\'s job, in a new process.\nPrompts are tokenized here (chat template, enable_thinking off unless think=True) and passed as token ids, so no double BOS.\nA prompt longer than max_len - max_tokens comes back as None and is counted in m.overflow (never crashes the batch).\nText after </think> is kept when a model still thinks. m.ntok_in / m.ntok_out count tokens of every call.\n"""\nimport os\nimport time\n\n# ChatML, for tokenizers that ship no chat template (vilm/vinallama-2.7b-chat): the format its model card describes\nCHATML = ("{% for m in messages %}{{ \'<|im_start|>\' + m[\'role\'] + \'\\n\' + m[\'content\'] + \'<|im_end|>\\n\' }}{% endfor %}"\n          "{% if add_generation_prompt %}{{ \'<|im_start|>assistant\\n\' }}{% endif %}")\n\n\nclass LLM:\n    def __init__(self, repo, max_len=16384, tp=1, think=False, **kw):\n        self.repo, self.max_len, self.think = repo, max_len, think\n        self.overflow = self.ntok_in = self.ntok_out = 0\n        self.backend = os.environ.get("LLM_BACKEND", "vllm")\n        t0 = time.time()\n        if self.backend == "vllm":\n            from vllm import LLM as V\n            self.eng = V(model=repo, dtype="float16", max_model_len=max_len, tensor_parallel_size=tp, seed=0,\n                         gpu_memory_utilization=float(os.environ.get("GPU_UTIL", "0.90")), trust_remote_code=True, **kw)\n            self.tok = self.eng.get_tokenizer()\n        else:\n            import torch\n            import transformers\n            from transformers import AutoModelForCausalLM, AutoTokenizer\n            import transformers.integrations.sdpa_attention as sd\n            if hasattr(sd, "use_gqa_in_sdpa"):  # T4: enable_gqa only runs in the math kernel (heads x L x L scores, OOM): repeat K/V instead\n                sd.use_gqa_in_sdpa = lambda *a, **k: False\n            dkw = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"\n            self.tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)\n            cuda = torch.cuda.is_available()\n            load = lambda remote: AutoModelForCausalLM.from_pretrained(repo, **{dkw: torch.float16 if cuda else torch.float32},\n                                                                       device_map="auto" if cuda else None, trust_remote_code=remote).eval()\n            try:\n                self.model = load(True)\n            except ImportError as e:  # repo code written for an older transformers (Phi-4-mini: LossKwargs); the built-in class works\n                print(f"[llm] remote code failed ({e}); loading the built-in transformers class", flush=True)\n                self.model = load(False)\n        if not getattr(self.tok, "chat_template", None):\n            self.tok.chat_template = CHATML\n            print(f"[llm] {repo}: no chat template, using ChatML", flush=True)\n        print(f"[llm] {repo} loaded with {self.backend} in {time.time() - t0:.0f}s", flush=True)\n\n    def ids(self, msgs):\n        kw = dict(tokenize=False, add_generation_prompt=True, enable_thinking=self.think)\n        try:\n            text = self.tok.apply_chat_template(msgs, **kw)\n        except Exception:  # templates without a system role: fold it into the first user turn\n            if msgs[0]["role"] != "system":\n                raise\n            text = self.tok.apply_chat_template([dict(msgs[1], content=msgs[0]["content"] + "\\n\\n" + msgs[1]["content"])] + msgs[2:], **kw)\n        if os.environ.get("STRIP_EMPTY_THINK") == "1":  # models fine-tuned by PIPE-T1-04 train.py on Qwen3.5 (no empty think block)\n            text = text.replace("<think>\\n\\n</think>\\n\\n", "")\n        return self.tok(text, add_special_tokens=False).input_ids\n\n    def gen(self, batch, max_tokens):\n        ids = [self.ids(m) for m in batch]\n        ok = [i for i, x in enumerate(ids) if len(x) <= self.max_len - max_tokens]\n        self.overflow += len(ids) - len(ok)\n        self.ntok_in += sum(len(ids[i]) for i in ok)\n        out = [None] * len(batch)\n        if self.backend == "vllm":\n            from vllm import SamplingParams\n            sp = SamplingParams(temperature=0.0, max_tokens=max_tokens, seed=0)\n            res = self.eng.generate([{"prompt_token_ids": ids[i]} for i in ok], sp, use_tqdm=False)\n            for i, r in zip(ok, res):\n                out[i] = r.outputs[0].text\n                self.ntok_out += len(r.outputs[0].token_ids)\n        else:\n            import torch\n            bs = int(os.environ.get("HF_BATCH", "4"))\n            pad = self.tok.pad_token_id if self.tok.pad_token_id is not None else self.tok.eos_token_id\n            dev = next(self.model.parameters()).device\n            order = sorted(ok, key=lambda i: len(ids[i]))\n            for k in range(0, len(order), bs):\n                part = order[k:k + bs]\n                n = max(len(ids[i]) for i in part)\n                x = torch.tensor([[pad] * (n - len(ids[i])) + ids[i] for i in part], device=dev)\n                mask = torch.tensor([[0] * (n - len(ids[i])) + [1] * len(ids[i]) for i in part], device=dev)\n                with torch.no_grad():\n                    y = self.model.generate(input_ids=x, attention_mask=mask, max_new_tokens=max_tokens, do_sample=False, pad_token_id=pad)\n                for i, row in zip(part, y[:, n:].tolist()):\n                    row = [t for t in row if t != pad]\n                    self.ntok_out += len(row)\n                    out[i] = self.tok.decode(row, skip_special_tokens=True)\n        return [o.split("</think>")[-1].strip() if o is not None else None for o in out]\n',
    'multirun.py': '"""Run one worker process per model over the GPUs of a Kaggle T4 x2 box.\n\n    ONLY=tag,tag python -u multirun.py jobs.json worker.py [-- extra worker args]      (ONLY empty or unset: every job)\n\njobs.json: [{"tag": "q3-4b", "tp": 1, "env": {"HARNESS": "4"}, "cands": [{"repo": "...", "backend": "vllm", ...}, ...], ...}, ...]\n(env: extra environment of that job\'s worker processes)\nEach candidate runs as `python -u worker.py --job \'<job + candidate as json>\' extra...` in its own process (a failed vLLM\nengine leaves a dirty process); exit code 0 ends the job, anything else tries the next candidate (fp16 -> AWQ, vllm -> hf).\ntp 2 jobs run one at a time on both GPUs first; tp 1 jobs then run two at a time, one per GPU. No GPU: one at a time.\nAfter a candidate its Hugging Face cache folder is deleted (Kaggle disk) unless the next candidate uses the same repo, or KEEP_HF=1.\nEvery worker line is echoed with a [tag] prefix; a summary of which candidate won is written to multirun_summary.json.\n"""\nimport json\nimport os\nimport queue\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\nLOCK = threading.Lock()\n\n\ndef log(msg):\n    with LOCK:\n        print(msg, flush=True)\n\n\ndef ngpu():\n    try:\n        return len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())\n    except FileNotFoundError:\n        return 0\n\n\ndef run_job(job, worker, extra, gpus):\n    for k, cand in enumerate(job["cands"]):\n        spec = dict(job, **cand, cand_index=k)\n        spec.pop("cands")\n        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpus, PYTHONUNBUFFERED="1", LLM_BACKEND=cand.get("backend", "vllm"), **job.get("env", {}))\n        t0 = time.time()\n        log(f"[{job[\'tag\']}] start candidate {k}: {cand[\'repo\']} ({env[\'LLM_BACKEND\']}) on GPU \'{gpus}\'")\n        p = subprocess.Popen([sys.executable, "-u", worker, "--job", json.dumps(spec, ensure_ascii=False)] + extra,\n                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)\n        tail = []\n        for line in p.stdout:\n            line = line.rstrip()\n            tail = (tail + [line])[-15:]\n            if "it/s]" not in line and "s/it]" not in line:  # drop tqdm bars\n                log(f"[{job[\'tag\']}] {line}")\n        rc = p.wait()\n        nxt = job["cands"][k + 1]["repo"] if k + 1 < len(job["cands"]) else None\n        if os.environ.get("KEEP_HF") != "1" and (rc == 0 or nxt != cand["repo"]):  # keep the weights for a retry of the same repo\n            hub = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface")) / "hub"\n            shutil.rmtree(hub / ("models--" + cand["repo"].replace("/", "--")), ignore_errors=True)\n        log(f"[{job[\'tag\']}] candidate {k} rc {rc} after {time.time() - t0:.0f}s")\n        if rc == 0:\n            return dict(tag=job["tag"], repo=cand["repo"], cand_index=k, ok=True, sec=round(time.time() - t0))\n    return dict(tag=job["tag"], ok=False, tail=tail)\n\n\ndef main():\n    argv = sys.argv[1:]\n    extra = argv[argv.index("--") + 1:] if "--" in argv else []\n    argv = argv[:argv.index("--")] if "--" in argv else argv\n    jobs = json.load(open(argv[0], encoding="utf-8"))\n    worker = argv[1]\n    if os.environ.get("ONLY"):\n        keep = os.environ["ONLY"].split(",")\n        jobs = [j for j in jobs if j["tag"] in keep]\n    n = ngpu()\n    log(f"[multirun] {len(jobs)} jobs, {n} GPU(s): {[j[\'tag\'] for j in jobs]}")\n    done = []\n    if n < 2:\n        done = [run_job(j, worker, extra, "0" if n else "") for j in jobs]\n    else:\n        done = [run_job(j, worker, extra, "0,1") for j in jobs if j.get("tp", 1) > 1]\n        q = queue.Queue()\n        for j in jobs:\n            if j.get("tp", 1) == 1:\n                q.put(j)\n\n        def drain(g):\n            while True:\n                try:\n                    j = q.get_nowait()\n                except queue.Empty:\n                    return\n                done.append(run_job(j, worker, extra, g))\n\n        ts = [threading.Thread(target=drain, args=(str(g),)) for g in range(2)]\n        [t.start() for t in ts]\n        [t.join() for t in ts]\n    json.dump(done, open("multirun_summary.json", "w"), indent=1)\n    log("[multirun] summary: " + json.dumps([(d["tag"], d.get("repo"), d["ok"]) for d in done]))\n\n\nif __name__ == "__main__":\n    main()\n',
    'irlib.py': '"""IR of one quantitative environmental figure + deterministic comparison (GreenScan AG-1, GS-Bench v1).\n\nA model may fill the IR; every number, unit, period, scope and boundary check is done here, so the\nverdict always points at an arithmetic step (invariants 4-6 of ARCHITECTURE_INVARIANTS.md):\n  - one dimension differs (metric, value kind, status, period, scope, boundary, unit) -> N, never C;\n  - a target never proves a result;\n  - values match within the precision each side was published to.\nPure stdlib so the same file runs on Kaggle and in the product.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import asdict, dataclass, field\n\nMETRICS = {\n    "ghg_emissions": "phát thải khí nhà kính",\n    "energy_consumption": "tiêu thụ năng lượng",\n    "electricity_consumption": "tiêu thụ điện",\n    "renewable_energy": "năng lượng tái tạo (sản xuất hoặc sử dụng)",\n    "fuel_consumption": "tiêu thụ nhiên liệu (than, dầu, khí, sinh khối)",\n    "water_use": "nước khai thác / sử dụng / tiêu thụ",\n    "wastewater": "nước thải phát sinh / xử lý / tuần hoàn",\n    "waste_generated": "chất thải phát sinh",\n    "waste_recycled": "chất thải tái chế / tái sử dụng / thu hồi",\n    "hazardous_waste": "chất thải nguy hại",\n    "recycled_input": "nguyên liệu tái chế đưa vào sản xuất",\n    "air_pollutants": "bụi, SO2, NOx, khí thải khác",\n    "land_trees": "trồng cây, diện tích xanh, rừng",\n    "other_env": "chỉ số môi trường khác",\n}\nVALUE_KINDS = ("absolute", "intensity", "share", "change", "count")\nSTATUSES = ("actual", "target")\nSCOPES = ("s1", "s2", "s3", "s1+s2", "s1+s2+s3")\nBOUNDARIES = ("group", "parent", "facility", "unspecified")\n\nMULTIPLIERS = {"nghìn": 1e3, "ngàn": 1e3, "ngàn tỷ": 1e12, "nghìn tỷ": 1e12, "triệu": 1e6, "tỷ": 1e9, "tỉ": 1e9,\n               "thousand": 1e3, "million": 1e6, "billion": 1e9, "": 1.0}\n\n# unit text -> (dimension, factor to the dimension\'s base unit)\n_UNITS = [\n    (r"(kt|nghìn tấn) ?co2", ("ghg_t", 1e3)), (r"kg ?co2", ("ghg_t", 1e-3)), (r"(t|tấn) ?co2", ("ghg_t", 1.0)),\n    (r"twh", ("energy_kwh", 1e9)), (r"gwh", ("energy_kwh", 1e6)), (r"mwh", ("energy_kwh", 1e3)), (r"kwh", ("energy_kwh", 1.0)),\n    (r"toe", ("energy_kwh", 11_630.0)), (r"\\btj\\b", ("energy_kwh", 277_777.8)), (r"\\bgj\\b", ("energy_kwh", 277.7778)),\n    (r"\\bmwp?\\b", ("power_mw", 1.0)), (r"\\bkwp?\\b", ("power_mw", 1e-3)),\n    (r"m3|m³|khối", ("volume_m3", 1.0)), (r"lít|\\bl\\b", ("volume_m3", 1e-3)),\n    (r"tấn|\\bt\\b|tons?|tonnes?", ("mass_t", 1.0)), (r"\\bkg\\b", ("mass_t", 1e-3)),\n    (r"%|phần trăm", ("percent", 1.0)), (r"\\bha\\b|hecta|héc", ("area_ha", 1.0)), (r"cây|trees?", ("count_tree", 1.0)),\n]\n\n\ndef unit_dimension(unit_text: str) -> tuple[str, float]:\n    """Dimension and factor of a unit; an intensity keeps its denominator (\'ghg_t/tấn thép\')."""\n    text = (unit_text or "").strip().lower()\n    num, _, den = text.partition("/")\n    dim, factor = next(((d, f) for pattern, (d, f) in _UNITS if re.search(pattern, num)), (num or "none", 1.0))\n    if den:\n        dim += "/" + re.sub(r"\\s+", " ", den.strip())\n    return dim, factor\n\n\ndef parse_number(text: str) -> tuple[float, float]:\n    """Value of a written number and half its last written digit (the published precision).\n\n    Vietnamese writes 22.540.603 and 23,5; some reports use 1,234.5. A separator followed by exactly\n    three digits is read as a thousands separator, any other as the decimal mark.\n    """\n    s = re.sub(r"\\s", "", text or "")\n    if not re.fullmatch(r"[\\d.,]+", s) or not re.search(r"\\d", s):\n        raise ValueError(f"not a number: {text!r}")\n    if "." in s and "," in s:\n        dec = "," if s.rfind(",") > s.rfind(".") else "."\n        integer, decimals = s.replace("." if dec == "," else ",", "").split(dec)\n    elif s.count(".") > 1 or s.count(",") > 1:\n        integer, decimals = re.sub(r"[.,]", "", s), ""\n    elif "." in s or "," in s:\n        integer, decimals = re.split(r"[.,]", s)\n        if len(decimals) == 3:\n            integer, decimals = integer + decimals, ""\n    else:\n        integer, decimals = s, ""\n    value = float(f"{integer or 0}.{decimals or 0}")\n    return value, 0.5 * 10 ** (-len(decimals))\n\n\n@dataclass\nclass IR:\n    metric: str\n    value_text: str\n    unit_text: str = ""\n    multiplier_text: str = ""\n    value_kind: str = "absolute"\n    change_direction: str | None = None  # decrease | increase, for value_kind == change\n    baseline_year: int | None = None\n    period: int | None = None\n    status: str = "actual"\n    scope: str | None = None\n    boundary: str = "unspecified"\n    metric_text: str = ""\n    subject_text: str = ""\n    extra: dict = field(default_factory=dict)\n\n    @classmethod\n    def from_dict(cls, d: dict) -> IR:\n        known = {k: d.get(k) for k in cls.__dataclass_fields__ if k in d and k != "extra"}\n        ir = cls(**known)\n        ir.boundary = ir.boundary or "unspecified"\n        ir.status = ir.status or "actual"\n        ir.value_kind = ir.value_kind or "absolute"\n        for name in ("period", "baseline_year"):\n            v = getattr(ir, name)\n            setattr(ir, name, int(v) if v not in (None, "", "null") and str(v).isdigit() else None)\n        return ir\n\n    def as_dict(self) -> dict:\n        return asdict(self)\n\n    def quantity(self) -> tuple[float, float, str]:\n        """Value and tolerance in the dimension\'s base unit, and the dimension."""\n        value, half = parse_number(self.value_text)\n        mult = MULTIPLIERS.get((self.multiplier_text or "").strip().lower(), 1.0)\n        dim, factor = unit_dimension(self.unit_text)\n        return value * mult * factor, half * mult * factor, dim\n\n\ndef comparable(claim: IR, evidence: IR) -> tuple[bool, str]:\n    """Same metric, value kind, status, period, scope, boundary and unit dimension (invariant 4)."""\n    checks = [\n        ("chỉ số", claim.metric, evidence.metric),\n        ("loại giá trị", claim.value_kind, evidence.value_kind),\n        ("thực hiện / mục tiêu", claim.status, evidence.status),\n        ("kỳ", claim.period, evidence.period),\n        ("phạm vi phát thải", claim.scope, evidence.scope),\n    ]\n    if claim.value_kind == "change":\n        checks.append(("năm gốc", claim.baseline_year, evidence.baseline_year))\n    if "unspecified" not in (claim.boundary, evidence.boundary):\n        checks.append(("ranh giới tổ chức", claim.boundary, evidence.boundary))\n    for name, a, b in checks:\n        if a != b:\n            return False, f"khác {name}: {a} / {b}"\n    try:\n        dim_c, dim_e = claim.quantity()[2], evidence.quantity()[2]\n    except ValueError as exc:\n        return False, f"không đọc được số: {exc}"\n    if dim_c != dim_e:\n        return False, f"khác đơn vị: {dim_c} / {dim_e}"\n    return True, "so được"\n\n\ndef compare(claim: IR, evidence: IR) -> tuple[str, str]:\n    """S (khớp), C (mâu thuẫn) or N (không so được), with the arithmetic or the reason."""\n    ok, why = comparable(claim, evidence)\n    if not ok:\n        return "N", why\n    if claim.value_kind == "change" and claim.change_direction != evidence.change_direction:\n        return "C", f"ngược chiều: {claim.change_direction} / {evidence.change_direction}"\n    vc, tc, dim = claim.quantity()\n    ve, te, _ = evidence.quantity()\n    gap = abs(vc - ve)\n    label = "S" if gap <= tc + te else "C"\n    return label, f"|{vc:g} − {ve:g}| = {gap:g} {dim}; dung sai {tc + te:g}"\n\n\ndef decide(claim: IR, evidence: list[IR]) -> tuple[str, list[tuple[str, str]]]:\n    """Verdict over every evidence figure: S if some comparable figure matches and none conflicts,\n    C if comparable figures only conflict, N otherwise (nothing comparable, or mixed: a person reads it)."""\n    steps = [compare(claim, e) for e in evidence]\n    labels = {label for label, _ in steps if label != "N"}\n    return (labels.pop() if len(labels) == 1 else "N"), steps\n',
    'ag1_worker.py': '"""AG-1 worker: one local LLM reads every claim and evidence text of GS-Bench v1 into IR figures; irlib decides.\n\n    python -u ag1_worker.py --job \'{"tag": "q3-4b-inst", "repo": "Qwen/Qwen3-4B-Instruct-2507", "tp": 1, "max_len": 4096}\'\n\nRuns inside a Kaggle notebook built by build_nb.py (files embedded next to it: llm.py from ZAIC Experiment/common,\nirlib.py, gs_bench_v1.jsonl). Greedy, thinking off. The model only fills the IR; S / C / N comes from irlib.decide\n(arithmetic by code, invariant 5). Output: out/<tag>_texts.jsonl (raw + parsed figures per unique text) and\nout/<tag>_cases.jsonl (verdict per case), plus a one-line summary in the log.\nEnv: LIMIT=n (first n cases, smoke), MAX_TOKENS (default 768).\n"""\nimport argparse, hashlib, json, os, re, time\nfrom pathlib import Path\n\nfrom irlib import IR, METRICS, SCOPES, VALUE_KINDS, decide\n\nPROMPT = """Bạn trích số liệu môi trường từ MỘT câu trong báo cáo của doanh nghiệp Việt Nam.\nVới MỖI con số mô tả một chỉ số môi trường (không lấy số năm, số điều luật, số thứ tự, số tiền), trả về một đối tượng:\n- "metric": một mã trong {metrics}\n- "metric_text": cụm từ gọi tên chỉ số, chép NGUYÊN VĂN từ câu\n- "value_text": con số, chép NGUYÊN VĂN từ câu (giữ dấu chấm, phẩy), không kèm chữ\n- "multiplier_text": "nghìn", "triệu", "tỷ" nếu câu viết chữ đó ngay sau số, ngược lại ""\n- "unit_text": đơn vị nguyên văn (vd "tCO2e", "kWh", "m3", "tấn", "%", "tCO2e/tấn sản phẩm"); "" nếu không có\n- "value_kind": một trong {kinds} (share = tỷ lệ phần trăm của một tổng; change = mức tăng/giảm so với kỳ trước hoặc năm gốc; intensity = trên một đơn vị sản phẩm/doanh thu)\n- "change_direction": "decrease" | "increase" | null (chỉ khi value_kind = change)\n- "baseline_year": năm gốc nếu là change và câu có nêu, ngược lại null\n- "period": năm của số liệu nếu câu NÊU RÕ; null nếu câu không nêu\n- "status": "actual" (đã thực hiện) | "target" (mục tiêu, kế hoạch, cam kết)\n- "scope": một trong {scopes} nếu là phát thải khí nhà kính và câu nêu phạm vi, ngược lại null\n- "boundary": "group" (tập đoàn, hợp nhất) | "parent" (công ty mẹ) | "facility" (nhà máy, cơ sở cụ thể) | "unspecified"\nCâu không có số liệu môi trường thì trả về danh sách rỗng.\n\nCÂU: {sentence}\n\nChỉ trả lời đúng một đối tượng JSON: {{"figures": [ ... ]}}"""\n\n\ndef parse(text):\n    """First JSON object in the reply; None when there is none."""\n    if not text:\n        return None\n    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)\n    start = text.find("{")\n    while start != -1:\n        depth = 0\n        for i in range(start, len(text)):\n            depth += {"{": 1, "}": -1}.get(text[i], 0)\n            if depth == 0:\n                try:\n                    obj = json.loads(text[start:i + 1])\n                    return obj if isinstance(obj, dict) else None\n                except json.JSONDecodeError:\n                    break\n        start = text.find("{", start + 1)\n    return None\n\n\ndef to_irs(figures):\n    out = []\n    for f in figures or []:\n        if isinstance(f, dict) and f.get("value_text") not in (None, ""):\n            try:\n                ir = IR.from_dict({**f, "value_text": str(f["value_text"])})\n                ir.quantity()  # unreadable number -> dropped\n                out.append(ir)\n            except (TypeError, ValueError):\n                pass\n    return out\n\n\ndef verdict(claim_irs, ev_irs):\n    if not claim_irs:\n        return "N", ["không đọc được số liệu trong tuyên bố"]\n    labels, why = [], []\n    for c in claim_irs:\n        label, steps = decide(c, ev_irs)\n        labels.append(label)\n        why.append(f"{c.value_text} {c.unit_text}: {label} " + "; ".join(s[1] for s in steps)[:300])\n    if "C" in labels:\n        return "C", why\n    return ("S" if all(x == "S" for x in labels) else "N"), why\n\n\ndef main():\n    ap = argparse.ArgumentParser(); ap.add_argument("--job")\n    job = json.loads(ap.parse_args().job)\n    tag = job["tag"]\n    cases = [json.loads(line) for line in open("gs_bench_v1.jsonl", encoding="utf-8")]\n    if os.environ.get("LIMIT"):\n        cases = cases[: int(os.environ["LIMIT"])]\n    texts = {}\n    for c in cases:\n        for t in (c["claim"], c["evidence"]):\n            texts.setdefault(hashlib.sha1(t.encode()).hexdigest()[:16], t)\n    for path in filter(None, os.environ.get("EXTRA_INPUT", "").split(",")):  # more sentences to read, no cases (second reader)\n        for r in map(json.loads, open(path, encoding="utf-8")):\n            texts.setdefault(hashlib.sha1(r["text"].encode()).hexdigest()[:16], r["text"])\n    print(f"[{tag}] {len(cases)} cases, {len(texts)} unique texts, repo {job[\'repo\']}", flush=True)\n    from llm import LLM\n    t0 = time.time()\n    m = LLM(job["repo"], max_len=job.get("max_len", 4096), tp=job.get("tp", 1), **job.get("kw", {}))\n    load_s = time.time() - t0\n    keys = list(texts)\n    prompts = [[{"role": "user", "content": PROMPT.format(metrics=list(METRICS), kinds=list(VALUE_KINDS), scopes=list(SCOPES),\n                                                         sentence=texts[k])}] for k in keys]\n    t1 = time.time()\n    raw = []\n    for i in range(0, len(prompts), 64):\n        raw += m.gen(prompts[i:i + 64], max_tokens=int(os.environ.get("MAX_TOKENS", "768")))\n        print(f"[{tag}] generated {len(raw)}/{len(prompts)}", flush=True)\n    gen_s = time.time() - t1\n    Path("out").mkdir(exist_ok=True)\n    irs, bad = {}, 0\n    with open(f"out/{tag}_texts.jsonl", "w", encoding="utf-8") as fh:\n        for k, r in zip(keys, raw):\n            obj = parse(r)\n            bad += obj is None\n            figs = (obj or {}).get("figures") if isinstance((obj or {}).get("figures"), list) else []\n            irs[k] = to_irs(figs)\n            fh.write(json.dumps({"key": k, "text": texts[k], "raw": r, "figures": [x.as_dict() for x in irs[k]]}, ensure_ascii=False) + "\\n")\n    right = 0\n    with open(f"out/{tag}_cases.jsonl", "w", encoding="utf-8") as fh:\n        for c in cases:\n            kc, ke = (hashlib.sha1(t.encode()).hexdigest()[:16] for t in (c["claim"], c["evidence"]))\n            label, why = verdict(irs[kc], irs[ke])\n            right += label in c["labels"]\n            fh.write(json.dumps({"id": c["id"], "label": label, "why": why}, ensure_ascii=False) + "\\n")\n    summary = {"tag": tag, "repo": job["repo"], "cases": len(cases), "right": right, "unparsed_texts": bad,\n               "load_s": round(load_s), "gen_s": round(gen_s), "ntok_in": getattr(m, "ntok_in", None), "ntok_out": getattr(m, "ntok_out", None)}\n    json.dump(summary, open(f"out/{tag}_summary.json", "w"), ensure_ascii=False)\n    print(f"[{tag}] SUMMARY {json.dumps(summary, ensure_ascii=False)}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
    'gs_bench_v1.jsonl': '{"id": "ba-F01", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khớp sau làm tròn", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty là khoảng 23,5 triệu tCO2e.", "evidence": "Bảng 5.2 — Phát thải khí nhà kính năm 2025: Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e; Tổng phạm vi 1 và 2: 23.474.480 tCO2e.", "labels": ["S"]}\n{"id": "ba-F02", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khớp sau làm tròn", "claim": "Phát thải khí nhà kính phạm vi 2 của Công ty năm 2025 là 933.876 tCO2e.", "evidence": "Bảng 5.2 — Phát thải khí nhà kính năm 2025: Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e; Tổng phạm vi 1 và 2: 23.474.480 tCO2e.", "labels": ["S"]}\n{"id": "ba-F03", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "lệch một con số", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty là 21,4 triệu tCO2e.", "evidence": "Bảng 5.2 — Phát thải khí nhà kính năm 2025: Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e; Tổng phạm vi 1 và 2: 23.474.480 tCO2e.", "labels": ["C"]}\n{"id": "ba-F04", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "lệch một con số", "claim": "Phát thải khí nhà kính phạm vi 2 của Công ty năm 2025 là 833.876 tCO2e.", "evidence": "Bảng 5.2 — Phát thải khí nhà kính năm 2025: Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e; Tổng phạm vi 1 và 2: 23.474.480 tCO2e.", "labels": ["C"]}\n{"id": "ba-F05", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "lệch một con số", "claim": "Năm 2024, nhà máy điện nhiệt dư đáp ứng 90% nhu cầu điện của khu liên hợp.", "evidence": "Năm 2024, nhà máy điện nhiệt dư đáp ứng 80% nhu cầu điện của khu liên hợp.", "labels": ["C"]}\n{"id": "ba-F06", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "tính lại % thay đổi", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 của Công ty giảm 20% so với năm 2020.", "evidence": "Phát thải khí nhà kính phạm vi 1 của Công ty: năm 2020 là 1.000.000 tCO2e; năm 2025 là 800.000 tCO2e.", "labels": ["S"]}\n{"id": "ba-F07", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "tính lại % thay đổi", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 của Công ty giảm 20% so với năm 2020.", "evidence": "Phát thải khí nhà kính phạm vi 1 của Công ty: năm 2020 là 1.000.000 tCO2e; năm 2025 là 850.000 tCO2e.", "labels": ["C"]}\n{"id": "ba-F08", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "tính lại % thay đổi", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 của Công ty giảm 20% so với năm 2020.", "evidence": "Phát thải khí nhà kính phạm vi 1 của Công ty: năm 2020 là 1.000.000 tCO2e; năm 2025 là 1.050.000 tCO2e.", "labels": ["C"]}\n{"id": "ba-F09", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác cơ sở đo", "claim": "Hoạt động sản xuất gang thép chiếm hơn 99% tổng phát thải khí nhà kính của Tập đoàn.", "evidence": "Năm 2025, sản lượng thép thô của Tập đoàn tăng 24% so với năm 2024.", "labels": ["N"]}\n{"id": "ba-F10", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác cơ sở đo", "claim": "Cường độ phát thải khí nhà kính năm 2025 của Công ty là 1,9 tCO2e/tấn thép.", "evidence": "Bảng 5.2 — Phát thải khí nhà kính năm 2025: Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e; Tổng phạm vi 1 và 2: 23.474.480 tCO2e.", "labels": ["N"]}\n{"id": "ba-F11", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác cơ sở đo", "claim": "Tổng phát thải khí nhà kính của Công ty năm 2025 giảm 5% so với năm 2024.", "evidence": "Cường độ phát thải khí nhà kính (tCO2e/tấn thép) của Công ty năm 2025 giảm 5% so với năm 2024.", "labels": ["N"]}\n{"id": "ba-F12", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác phạm vi phát thải", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 của Công ty giảm 30% so với năm 2020.", "evidence": "Năm 2025, phát thải khí nhà kính phạm vi 1, 2 và 3 của Công ty giảm 8% so với năm 2020.", "labels": ["N"]}\n{"id": "ba-F13", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác phạm vi phát thải", "claim": "Phát thải khí nhà kính phạm vi 1 và 2 của Công ty năm 2025 là 23,5 triệu tCO2e.", "evidence": "Phát thải khí nhà kính phạm vi 1 của Công ty năm 2025 là 22.540.603 tCO2e.", "labels": ["N"]}\n{"id": "ba-F14", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác kỳ", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 của Công ty là 22,5 triệu tCO2e.", "evidence": "Năm 2024, phát thải khí nhà kính phạm vi 1 của Công ty là 22,5 triệu tCO2e.", "labels": ["N"]}\n{"id": "ba-F15", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác kỳ", "claim": "Năm 2025, lượng nước sử dụng của Công ty giảm 12% so với năm 2024.", "evidence": "Năm 2023, lượng nước sử dụng của Công ty giảm 12% so với năm 2022.", "labels": ["N"]}\n{"id": "ba-F16", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "mục tiêu hay thực hiện", "claim": "Công ty đã giảm 30% phát thải khí nhà kính phạm vi 1 và 2 so với năm 2020.", "evidence": "Mục tiêu đến năm 2030: giảm 30% phát thải khí nhà kính phạm vi 1 và 2 so với năm 2020.", "labels": ["N"]}\n{"id": "ba-F17", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "mục tiêu hay thực hiện", "claim": "Công ty đặt mục tiêu giảm 30% phát thải khí nhà kính phạm vi 1 và 2 vào năm 2030 so với năm 2020.", "evidence": "Mục tiêu đến năm 2030: giảm 30% phát thải khí nhà kính phạm vi 1 và 2 so với năm 2020.", "labels": ["S"]}\n{"id": "ba-F18", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "mục tiêu hay thực hiện", "claim": "Năm 2025, tỷ lệ phế liệu tái chế đưa vào sản xuất của Công ty đạt 30%.", "evidence": "Năm 2025, tỷ lệ phế liệu tái chế đưa vào sản xuất của Công ty đạt 18%, thấp hơn mục tiêu 30%.", "labels": ["C"]}\n{"id": "ba-F19", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác ranh giới tổ chức", "claim": "Phát thải khí nhà kính phạm vi 1 và 2 hợp nhất của Tập đoàn năm 2025 là 23,5 triệu tCO2e.", "evidence": "Phát thải khí nhà kính phạm vi 1 và 2 của Khu liên hợp Dung Quất năm 2025 là 12.300.000 tCO2e.", "labels": ["N"]}\n{"id": "ba-F20", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác ranh giới tổ chức", "claim": "Phát thải khí nhà kính phạm vi 1 và 2 của công ty mẹ năm 2025 là 1,2 triệu tCO2e.", "evidence": "Phát thải khí nhà kính phạm vi 1 và 2 hợp nhất toàn Tập đoàn năm 2025 là 23,47 triệu tCO2e.", "labels": ["N"]}\n{"id": "ba-F21", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác đơn vị / hệ số", "claim": "Năm 2025, Công ty tiêu thụ 5,6 triệu MWh điện.", "evidence": "Năm 2025, Công ty tiêu thụ 5,6 triệu tấn than.", "labels": ["N"]}\n{"id": "ba-F22", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác đơn vị / hệ số", "claim": "Năm 2025, Công ty sử dụng 15 triệu m3 nước.", "evidence": "Lượng nước sử dụng của Công ty năm 2025: 15.000 m3.", "labels": ["C"]}\n{"id": "ba-F23", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác đơn vị / hệ số", "claim": "Năm 2025, Công ty sử dụng 15 triệu m3 nước.", "evidence": "Lượng nước sử dụng của Công ty năm 2025: 15.000.000 m3.", "labels": ["S"]}\n{"id": "ba-F24", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "dung sai theo độ chính xác", "claim": "Tỷ lệ năng lượng tái tạo trong tổng năng lượng tiêu thụ của Công ty năm 2025 đạt 12%.", "evidence": "Tỷ lệ năng lượng tái tạo trong tổng năng lượng tiêu thụ của Công ty năm 2025: 12,4%.", "labels": ["S"]}\n{"id": "ba-F25", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "dung sai theo độ chính xác", "claim": "Tỷ lệ năng lượng tái tạo trong tổng năng lượng tiêu thụ của Công ty năm 2025 đạt 12,0%.", "evidence": "Tỷ lệ năng lượng tái tạo trong tổng năng lượng tiêu thụ của Công ty năm 2025: 12,4%.", "labels": ["C"]}\n{"id": "ba-F26", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "cộng lại bảng", "claim": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty năm 2025 là 23.474.480 tCO2e.", "evidence": "Phát thải khí nhà kính năm 2025 của Công ty — Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e.", "labels": ["S"]}\n{"id": "ba-F27", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "cộng lại bảng", "claim": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty năm 2025 là 24.474.480 tCO2e.", "evidence": "Phát thải khí nhà kính năm 2025 của Công ty — Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e.", "labels": ["C"]}\n{"id": "ba-F28", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "bẫy từ phủ định", "claim": "Năm 2024, Tập đoàn giảm phát thải khí nhà kính tại các nhà máy thép.", "evidence": "Năm 2024, nhà máy thép áp dụng công nghệ thiêu kết mới giúp giảm phát thải khí nhà kính.", "labels": ["N", "S"]}\n{"id": "ba-F29", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "bẫy từ phủ định", "claim": "Năm 2024, Tập đoàn giảm phát thải khí nhà kính tại các nhà máy thép.", "evidence": "Năm 2024 không có sự cố môi trường nào tại các nhà máy thép; phát thải khí nhà kính giảm so với năm trước.", "labels": ["N", "S"]}\n{"id": "ba-F30", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "không có bằng chứng liên quan", "claim": "Năm 2025, 100% nước thải của Công ty được xử lý đạt quy chuẩn trước khi xả ra môi trường.", "evidence": "Năm 2025, Công ty tổ chức 12 khoá đào tạo an toàn lao động cho 3.000 lượt người lao động.", "labels": ["N"]}\n{"id": "ba-F31", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "không có bằng chứng liên quan", "claim": "Công ty đã đạt trung hòa carbon trong năm 2025.", "evidence": "Năm 2025, Công ty trồng 50.000 cây xanh tại các khu công nghiệp.", "labels": ["N"]}\n{"id": "ba-F32", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "cùng số, khác chỉ số", "claim": "Năm 2025, Công ty tự chủ 90% nhu cầu điện cho sản xuất thép.", "evidence": "Năm 2025, Công ty hoàn thành 90% kế hoạch kiểm kê khí nhà kính.", "labels": ["N"]}\n{"id": "ba-N01", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 toàn Tập đoàn là 23,47 triệu tCO2e.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["S"]}\n{"id": "ba-N02", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 toàn Tập đoàn là 23.369.500 tCO2e.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["C"]}\n{"id": "ba-N03", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, mảng thép phát thải 23.369.500 tCO2e, chiếm 99,55% tổng phát thải khí nhà kính của Tập đoàn.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["S"]}\n{"id": "ba-N04", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, mảng nông nghiệp chiếm 0,39% tổng phát thải khí nhà kính của Tập đoàn.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["S"]}\n{"id": "ba-N05", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, mảng nông nghiệp phát thải 933.876 tCO2e.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["C"]}\n{"id": "ba-N06", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 3 toàn Tập đoàn là 23,47 triệu tCO2e.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["N"]}\n{"id": "ba-N08", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 2 toàn Tập đoàn là 0,93 triệu tCO2e.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["S"]}\n{"id": "ba-N09", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "chữ dính cột", "claim": "Năm 2025, Tập đoàn hoàn thành 90% kế hoạch kiểm kê khí nhà kính.", "evidence": "9\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\nĐóng góp gần 13.000 tỷ đồng vào \\nngân sách Nhà nước  \\nĐóng góp hơn 120 tỷ đồng cho các \\nchương trình phúc lợi xã hội.\\nMÔI TRƯỜNG\\nXÃ HỘI\\nQUẢN TRỊ\\nHoàn thành kiểm kê khí nhà kính theo tiêu \\nchuẩn ISO 14064-1:2018 cho toàn Tập \\nđoàn và được BSI thẩm định số liệu.  \\nTái chế, tái sử dụng chất thải rắn\\nnhóm thép và nông nghiệp đạt trên 99% \\nSố giờ đào tạo trung bình cho mỗi nhân \\nviên là 38,52 giờ mỗi năm \\n22% thành viên\\nHội đồng Quản trị là thành viên độc lập.  \\nĐiểm nổi bật về\\nphát triển bền vững năm 2025\\nGiải thưởng\\nvà vinh danh năm 2025\\nThành viên của\\ncác Hiệp hội ngành nghề\\nHoạt động kinh doanh\\ntại Tập đoàn\\nPHẦN 3\\nTỔNG QUAN\\nVỀ TẬP ĐOÀN \\nHÒA PHÁT\\nĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025\\n3.1\\nTự chủ 90% nhu cầu điện\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTái sử dụng 98% nước\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTỷ lệ phụ nữ ở các cấp quản lý trong toàn \\nTập đoàn đạt 35,85% vào năm 2025\\n", "labels": ["C", "N"]}\n{"id": "ba-N10", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "chữ dính cột", "claim": "Khu liên hợp Gang thép Hòa Phát Dung Quất tự chủ 90% nhu cầu điện.", "evidence": "9\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\nĐóng góp gần 13.000 tỷ đồng vào \\nngân sách Nhà nước  \\nĐóng góp hơn 120 tỷ đồng cho các \\nchương trình phúc lợi xã hội.\\nMÔI TRƯỜNG\\nXÃ HỘI\\nQUẢN TRỊ\\nHoàn thành kiểm kê khí nhà kính theo tiêu \\nchuẩn ISO 14064-1:2018 cho toàn Tập \\nđoàn và được BSI thẩm định số liệu.  \\nTái chế, tái sử dụng chất thải rắn\\nnhóm thép và nông nghiệp đạt trên 99% \\nSố giờ đào tạo trung bình cho mỗi nhân \\nviên là 38,52 giờ mỗi năm \\n22% thành viên\\nHội đồng Quản trị là thành viên độc lập.  \\nĐiểm nổi bật về\\nphát triển bền vững năm 2025\\nGiải thưởng\\nvà vinh danh năm 2025\\nThành viên của\\ncác Hiệp hội ngành nghề\\nHoạt động kinh doanh\\ntại Tập đoàn\\nPHẦN 3\\nTỔNG QUAN\\nVỀ TẬP ĐOÀN \\nHÒA PHÁT\\nĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025\\n3.1\\nTự chủ 90% nhu cầu điện\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTái sử dụng 98% nước\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTỷ lệ phụ nữ ở các cấp quản lý trong toàn \\nTập đoàn đạt 35,85% vào năm 2025\\n", "labels": ["S"]}\n{"id": "ba-N11", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "chữ dính cột", "claim": "Khu liên hợp Gang thép Hòa Phát Dung Quất tái sử dụng 90% nước.", "evidence": "9\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\nĐóng góp gần 13.000 tỷ đồng vào \\nngân sách Nhà nước  \\nĐóng góp hơn 120 tỷ đồng cho các \\nchương trình phúc lợi xã hội.\\nMÔI TRƯỜNG\\nXÃ HỘI\\nQUẢN TRỊ\\nHoàn thành kiểm kê khí nhà kính theo tiêu \\nchuẩn ISO 14064-1:2018 cho toàn Tập \\nđoàn và được BSI thẩm định số liệu.  \\nTái chế, tái sử dụng chất thải rắn\\nnhóm thép và nông nghiệp đạt trên 99% \\nSố giờ đào tạo trung bình cho mỗi nhân \\nviên là 38,52 giờ mỗi năm \\n22% thành viên\\nHội đồng Quản trị là thành viên độc lập.  \\nĐiểm nổi bật về\\nphát triển bền vững năm 2025\\nGiải thưởng\\nvà vinh danh năm 2025\\nThành viên của\\ncác Hiệp hội ngành nghề\\nHoạt động kinh doanh\\ntại Tập đoàn\\nPHẦN 3\\nTỔNG QUAN\\nVỀ TẬP ĐOÀN \\nHÒA PHÁT\\nĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025\\n3.1\\nTự chủ 90% nhu cầu điện\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTái sử dụng 98% nước\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTỷ lệ phụ nữ ở các cấp quản lý trong toàn \\nTập đoàn đạt 35,85% vào năm 2025\\n", "labels": ["C"]}\n{"id": "ba-N12", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "chữ dính cột", "claim": "Năm 2025, tỷ lệ tái chế, tái sử dụng chất thải rắn của nhóm thép và nông nghiệp đạt trên 99%.", "evidence": "9\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\nĐóng góp gần 13.000 tỷ đồng vào \\nngân sách Nhà nước  \\nĐóng góp hơn 120 tỷ đồng cho các \\nchương trình phúc lợi xã hội.\\nMÔI TRƯỜNG\\nXÃ HỘI\\nQUẢN TRỊ\\nHoàn thành kiểm kê khí nhà kính theo tiêu \\nchuẩn ISO 14064-1:2018 cho toàn Tập \\nđoàn và được BSI thẩm định số liệu.  \\nTái chế, tái sử dụng chất thải rắn\\nnhóm thép và nông nghiệp đạt trên 99% \\nSố giờ đào tạo trung bình cho mỗi nhân \\nviên là 38,52 giờ mỗi năm \\n22% thành viên\\nHội đồng Quản trị là thành viên độc lập.  \\nĐiểm nổi bật về\\nphát triển bền vững năm 2025\\nGiải thưởng\\nvà vinh danh năm 2025\\nThành viên của\\ncác Hiệp hội ngành nghề\\nHoạt động kinh doanh\\ntại Tập đoàn\\nPHẦN 3\\nTỔNG QUAN\\nVỀ TẬP ĐOÀN \\nHÒA PHÁT\\nĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025\\n3.1\\nTự chủ 90% nhu cầu điện\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTái sử dụng 98% nước\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTỷ lệ phụ nữ ở các cấp quản lý trong toàn \\nTập đoàn đạt 35,85% vào năm 2025\\n", "labels": ["S"]}\n{"id": "ba-N13", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "chữ dính cột", "claim": "Năm 2025, tỷ lệ phụ nữ ở các cấp quản lý trong toàn Tập đoàn đạt 22%.", "evidence": "9\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\nĐóng góp gần 13.000 tỷ đồng vào \\nngân sách Nhà nước  \\nĐóng góp hơn 120 tỷ đồng cho các \\nchương trình phúc lợi xã hội.\\nMÔI TRƯỜNG\\nXÃ HỘI\\nQUẢN TRỊ\\nHoàn thành kiểm kê khí nhà kính theo tiêu \\nchuẩn ISO 14064-1:2018 cho toàn Tập \\nđoàn và được BSI thẩm định số liệu.  \\nTái chế, tái sử dụng chất thải rắn\\nnhóm thép và nông nghiệp đạt trên 99% \\nSố giờ đào tạo trung bình cho mỗi nhân \\nviên là 38,52 giờ mỗi năm \\n22% thành viên\\nHội đồng Quản trị là thành viên độc lập.  \\nĐiểm nổi bật về\\nphát triển bền vững năm 2025\\nGiải thưởng\\nvà vinh danh năm 2025\\nThành viên của\\ncác Hiệp hội ngành nghề\\nHoạt động kinh doanh\\ntại Tập đoàn\\nPHẦN 3\\nTỔNG QUAN\\nVỀ TẬP ĐOÀN \\nHÒA PHÁT\\nĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025\\n3.1\\nTự chủ 90% nhu cầu điện\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTái sử dụng 98% nước\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTỷ lệ phụ nữ ở các cấp quản lý trong toàn \\nTập đoàn đạt 35,85% vào năm 2025\\n", "labels": ["C"]}\n{"id": "ba-N14", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Năm 2024, các nhà máy điện nhiệt dư tại hai khu liên hợp Hải Dương và Dung Quất phát 3,18 tỷ kWh điện, tăng 29% so với năm trước.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["S"]}\n{"id": "ba-N15", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Năm 2024, nhà máy điện nhiệt dư tại Dung Quất phát 3,18 tỷ kWh điện.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["C"]}\n{"id": "ba-N16", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Năm 2024, lượng điện phát của các nhà máy điện nhiệt dư thuộc hai khu liên hợp tăng 39% so với cùng kỳ.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["C"]}\n{"id": "ba-N17", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Năm 2024, Hòa Phát tự chủ 100% lượng điện cho sản xuất thép.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["C", "N"]}\n{"id": "ba-N18", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Công nghệ tuabin thu hồi năng lượng quạt gió lò cao (BPRT) giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương pháp truyền thống.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["S"]}\n{"id": "ba-N19", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Năm 2024, giải pháp thu hồi nhiệt dư giúp Hòa Phát giảm 50% phát thải CO2.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["N"]}\n{"id": "ba-N20", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Ngày 05/02/2025, nhà máy nhiệt điện tại Dung Quất đạt sản lượng điện lũy kế 10 tỷ kWh.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["S"]}\n{"id": "bb-2a31912f6d377a83-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tổng lượng nước sử dụng của Công ty là 34.325 m3.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2a31912f6d377a83-0-S1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S1 rounded", "claim": "Tổng lượng nước sử dụng của Công ty là 34,3 nghìn m3.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "34,3", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2a31912f6d377a83-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tổng lượng nước sử dụng của Công ty là 48.055 m3.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "48.055", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2a31912f6d377a83-0-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Tổng lượng nước sử dụng của Công ty là 34.325 nghìn m3.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2a31912f6d377a83-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tổng lượng nước sử dụng đạt 34.325 m3 vào năm 2030.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2a31912f6d377a83-0-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Tổng lượng nước sử dụng của Công ty là 34.325 m3/tấn sản phẩm.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2a31912f6d377a83-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước thải của Công ty là 34.325 m3.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-f5f8a5c13872aea7-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Mức tiêu thụ nước của Công ty giảm 12,8% so với năm 2024.", "evidence": "Mức tiêu thụ nước giảm : 12,8% (so với năm 2024: 4.878 m3) VÀ PHÁT THẢI HIỆU QUẢ • Phun nước làm ẩm đường vào những ngày khô hanh.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-f5f8a5c13872aea7-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Mức tiêu thụ nước của Công ty giảm 27,8% so với năm 2024.", "evidence": "Mức tiêu thụ nước giảm : 12,8% (so với năm 2024: 4.878 m3) VÀ PHÁT THẢI HIỆU QUẢ • Phun nước làm ẩm đường vào những ngày khô hanh.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "27,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-f5f8a5c13872aea7-0-C3", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C3 direction", "claim": "Mức tiêu thụ nước của Công ty tăng 12,8% so với năm 2024.", "evidence": "Mức tiêu thụ nước giảm : 12,8% (so với năm 2024: 4.878 m3) VÀ PHÁT THẢI HIỆU QUẢ • Phun nước làm ẩm đường vào những ngày khô hanh.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-f5f8a5c13872aea7-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Mức tiêu thụ nước giảm 12,8% vào năm 2030.", "evidence": "Mức tiêu thụ nước giảm : 12,8% (so với năm 2024: 4.878 m3) VÀ PHÁT THẢI HIỆU QUẢ • Phun nước làm ẩm đường vào những ngày khô hanh.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-f5f8a5c13872aea7-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước thải của Công ty giảm 12,8% so với năm 2024.", "evidence": "Mức tiêu thụ nước giảm : 12,8% (so với năm 2024: 4.878 m3) VÀ PHÁT THẢI HIỆU QUẢ • Phun nước làm ẩm đường vào những ngày khô hanh.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-9c78bf02decaf05d-0-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Năm 2021, mức tiêu thụ gas và xăng dầu của Công ty giảm 39% so với năm 2020.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["S"], "claim_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-0-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Năm 2021, mức tiêu thụ gas và xăng dầu của Công ty giảm 54% so với năm 2020.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "54", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-0-C3", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C3 direction", "claim": "Năm 2021, mức tiêu thụ gas và xăng dầu của Công ty tăng 39% so với năm 2020.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-0-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu mức tiêu thụ gas và xăng dầu giảm 39% vào năm 2030.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-1-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Năm 2021, mức tiêu thụ gas và xăng dầu của Công ty giảm 14% so với năm 2020.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["S"], "claim_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-1-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Năm 2021, mức tiêu thụ gas và xăng dầu của Công ty giảm 29% so với năm 2020.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "29", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-1-C3", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C3 direction", "claim": "Năm 2021, mức tiêu thụ gas và xăng dầu của Công ty tăng 14% so với năm 2020.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-1-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu mức tiêu thụ gas và xăng dầu giảm 14% vào năm 2030.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-270002ea10dc9ad7-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tổng lượng nước sử dụng của Công ty là 10.730 m3.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-270002ea10dc9ad7-0-S1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S1 rounded", "claim": "Tổng lượng nước sử dụng của Công ty là 10,7 nghìn m3.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "10,7", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-270002ea10dc9ad7-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tổng lượng nước sử dụng của Công ty là 8.584 m3.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "8.584", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-270002ea10dc9ad7-0-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Tổng lượng nước sử dụng của Công ty là 10.730 nghìn m3.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-270002ea10dc9ad7-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tổng lượng nước sử dụng đạt 10.730 m3 vào năm 2030.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-270002ea10dc9ad7-0-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Tổng lượng nước sử dụng của Công ty là 10.730 m3/tấn sản phẩm.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-270002ea10dc9ad7-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước thải của Công ty là 10.730 m3.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-9796bc7d982bb3ce-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Năm 2014, tổng tiêu thụ điện của Công ty giảm 6% so với năm 2013.", "evidence": "TIÊU THỤ TÀI NGUYÊN ĐIỆN, NƯỚC, GIẤY Năm 2014, tổng tiêu thụ điện khu vực văn phòng giảm 6% so với năm 2013, duy trì tiết kiệm điện nước, ổn định mức tiêu thụ điện nước cho sản xuất, lượng giấy sử dụng cho toàn bộ hoạt động của Công ty.", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-9796bc7d982bb3ce-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Năm 2014, tổng tiêu thụ điện của Công ty giảm 21% so với năm 2013.", "evidence": "TIÊU THỤ TÀI NGUYÊN ĐIỆN, NƯỚC, GIẤY Năm 2014, tổng tiêu thụ điện khu vực văn phòng giảm 6% so với năm 2013, duy trì tiết kiệm điện nước, ổn định mức tiêu thụ điện nước cho sản xuất, lượng giấy sử dụng cho toàn bộ hoạt động của Công ty.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "21", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-9796bc7d982bb3ce-0-C3", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C3 direction", "claim": "Năm 2014, tổng tiêu thụ điện của Công ty tăng 6% so với năm 2013.", "evidence": "TIÊU THỤ TÀI NGUYÊN ĐIỆN, NƯỚC, GIẤY Năm 2014, tổng tiêu thụ điện khu vực văn phòng giảm 6% so với năm 2013, duy trì tiết kiệm điện nước, ổn định mức tiêu thụ điện nước cho sản xuất, lượng giấy sử dụng cho toàn bộ hoạt động của Công ty.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-9796bc7d982bb3ce-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu tổng tiêu thụ điện giảm 6% vào năm 2030.", "evidence": "TIÊU THỤ TÀI NGUYÊN ĐIỆN, NƯỚC, GIẤY Năm 2014, tổng tiêu thụ điện khu vực văn phòng giảm 6% so với năm 2013, duy trì tiết kiệm điện nước, ổn định mức tiêu thụ điện nước cho sản xuất, lượng giấy sử dụng cho toàn bộ hoạt động của Công ty.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-9796bc7d982bb3ce-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Năm 2014, lượng điện từ năng lượng tái tạo của Công ty giảm 6% so với năm 2013.", "evidence": "TIÊU THỤ TÀI NGUYÊN ĐIỆN, NƯỚC, GIẤY Năm 2014, tổng tiêu thụ điện khu vực văn phòng giảm 6% so với năm 2013, duy trì tiết kiệm điện nước, ổn định mức tiêu thụ điện nước cho sản xuất, lượng giấy sử dụng cho toàn bộ hoạt động của Công ty.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng điện từ năng lượng tái tạo", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tổng lượng nước sử dụng của Công ty là 12.726 m3.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-S1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S1 rounded", "claim": "Tổng lượng nước sử dụng của Công ty là 12,7 nghìn m3.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "12,7", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tổng lượng nước sử dụng của Công ty là 15.908 m3.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "15.908", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Tổng lượng nước sử dụng của Công ty là 12.726 nghìn m3.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tổng lượng nước sử dụng đạt 12.726 m3 vào năm 2030.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Tổng lượng nước sử dụng của Công ty là 12.726 m3/tấn sản phẩm.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước thải của Công ty là 12.726 m3.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-d7a04b919bbe8cda-1-S0", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "S0 restate", "claim": "Lượng nước tiêu thụ của Công ty giảm 41% so với năm 2024.", "evidence": "nước sinh hoạt chiếm tỷ trọng lớn, khoảng 4.849 m3, trong khi nước uống tinh khiết So với năm 2024, lượng nước tiêu thụ ghi nhận xu hướng giảm đáng kể khoảng 41%, cho thấy hiệu quả của các giải pháp kiểm soát và tối ưu sử dụng nước trong toàn hệ thống.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 4}\n{"id": "bb-d7a04b919bbe8cda-1-C1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C1 value", "claim": "Lượng nước tiêu thụ của Công ty giảm 56% so với năm 2024.", "evidence": "nước sinh hoạt chiếm tỷ trọng lớn, khoảng 4.849 m3, trong khi nước uống tinh khiết So với năm 2024, lượng nước tiêu thụ ghi nhận xu hướng giảm đáng kể khoảng 41%, cho thấy hiệu quả của các giải pháp kiểm soát và tối ưu sử dụng nước trong toàn hệ thống.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "56", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 4}\n{"id": "bb-d7a04b919bbe8cda-1-C3", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C3 direction", "claim": "Lượng nước tiêu thụ của Công ty tăng 41% so với năm 2024.", "evidence": "nước sinh hoạt chiếm tỷ trọng lớn, khoảng 4.849 m3, trong khi nước uống tinh khiết So với năm 2024, lượng nước tiêu thụ ghi nhận xu hướng giảm đáng kể khoảng 41%, cho thấy hiệu quả của các giải pháp kiểm soát và tối ưu sử dụng nước trong toàn hệ thống.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 4}\n{"id": "bb-d7a04b919bbe8cda-1-N2", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N2 status", "claim": "Công ty đặt mục tiêu lượng nước tiêu thụ giảm 41% vào năm 2030.", "evidence": "nước sinh hoạt chiếm tỷ trọng lớn, khoảng 4.849 m3, trong khi nước uống tinh khiết So với năm 2024, lượng nước tiêu thụ ghi nhận xu hướng giảm đáng kể khoảng 41%, cho thấy hiệu quả của các giải pháp kiểm soát và tối ưu sử dụng nước trong toàn hệ thống.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 4}\n{"id": "bb-d7a04b919bbe8cda-1-N6", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N6 metric", "claim": "Lượng nước thải của Công ty giảm 41% so với năm 2024.", "evidence": "nước sinh hoạt chiếm tỷ trọng lớn, khoảng 4.849 m3, trong khi nước uống tinh khiết So với năm 2024, lượng nước tiêu thụ ghi nhận xu hướng giảm đáng kể khoảng 41%, cho thấy hiệu quả của các giải pháp kiểm soát và tối ưu sử dụng nước trong toàn hệ thống.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 4}\n{"id": "bb-db8841dc88c6b9be-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tổng lượng nước sử dụng của Công ty là 8.131 m3.", "evidence": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-db8841dc88c6b9be-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tổng lượng nước sử dụng của Công ty là 5.692 m3.", "evidence": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "5.692", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-db8841dc88c6b9be-0-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Tổng lượng nước sử dụng của Công ty là 8.131 nghìn m3.", "evidence": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-db8841dc88c6b9be-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tổng lượng nước sử dụng đạt 8.131 m3 vào năm 2030.", "evidence": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-db8841dc88c6b9be-0-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Tổng lượng nước sử dụng của Công ty là 8.131 m3/tấn sản phẩm.", "evidence": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-db8841dc88c6b9be-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước thải của Công ty là 8.131 m3.", "evidence": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Năm 2023, Điện năng tiêu thụ của Công ty là 33,55 triệu Kwh.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Năm 2023, Điện năng tiêu thụ của Công ty là 46,97 triệu Kwh.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "46,97", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-C2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C2 magnitude", "claim": "Năm 2023, Điện năng tiêu thụ của Công ty là 33,55 tỷ Kwh.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "tỷ", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-N1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N1 period", "claim": "Năm 2022, Điện năng tiêu thụ của Công ty là 33,55 triệu Kwh.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Điện năng tiêu thụ đạt 33,55 triệu Kwh vào năm 2030.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-N5", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N5 basis", "claim": "Năm 2023, Điện năng tiêu thụ của Công ty là 33,55 triệu Kwh/tấn sản phẩm.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh/tấn sản phẩm", "multiplier_text": "triệu", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Năm 2023, lượng điện từ năng lượng tái tạo của Công ty là 33,55 triệu Kwh.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng điện từ năng lượng tái tạo", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Năm 2025, nước thải của Công ty là 159.104 m3.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["S"], "claim_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-S1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S1 rounded", "claim": "Năm 2025, nước thải của Công ty là 159,1 nghìn m3.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["S"], "claim_ir": {"metric": "wastewater", "value_text": "159,1", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Năm 2025, nước thải của Công ty là 127.283 m3.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["C"], "claim_ir": {"metric": "wastewater", "value_text": "127.283", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-C2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C2 magnitude", "claim": "Năm 2025, nước thải của Công ty là 159.104 nghìn m3.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["C"], "claim_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-N1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N1 period", "claim": "Năm 2024, nước thải của Công ty là 159.104 m3.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2024, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu nước thải đạt 159.104 m3 vào năm 2030.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-N5", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N5 basis", "claim": "Năm 2025, nước thải của Công ty là 159.104 m3/tấn sản phẩm.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Năm 2025, lượng nước sử dụng của Công ty là 159.104 m3.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-631428f0da19badc-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Năng lượng xanh và sạch của Công ty đạt 87%.", "evidence": "Tỷ lệ sử dụng năng lượng xanh và sạch trong hoạt động sản xuất đạt 87%.", "labels": ["S"], "claim_ir": {"metric": "renewable_energy", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-631428f0da19badc-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Năng lượng xanh và sạch của Công ty đạt 72%.", "evidence": "Tỷ lệ sử dụng năng lượng xanh và sạch trong hoạt động sản xuất đạt 87%.", "labels": ["C"], "claim_ir": {"metric": "renewable_energy", "value_text": "72", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-631428f0da19badc-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu năng lượng xanh và sạch đạt 87% vào năm 2030.", "evidence": "Tỷ lệ sử dụng năng lượng xanh và sạch trong hoạt động sản xuất đạt 87%.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-631428f0da19badc-0-N6", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N6 metric", "claim": "Tổng lượng điện tiêu thụ của Công ty đạt 87%.", "evidence": "Tỷ lệ sử dụng năng lượng xanh và sạch trong hoạt động sản xuất đạt 87%.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng lượng điện tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-e14c97c34811ab03-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Mức tiêu thụ điện năng của Công ty giảm 10,1% so với năm 2016.", "evidence": "So với năm 2016 mức tiêu thụ điện năng tiêu giảm được 10,1% (năm 2016: 2.686.080 kW), một phần trong vấn đề giảm tiêu hao năng lượng là do có sự chuyển dịch về sản xuất sản phẩm sang 2 nhà máy Traphaco Hưng Yên và Traphaco CNC và áp dụng các biện pháp giảm tiêu hao năng lượng nhằm phát triển môi trường bền vững.", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-e14c97c34811ab03-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Mức tiêu thụ điện năng của Công ty giảm 25,1% so với năm 2016.", "evidence": "So với năm 2016 mức tiêu thụ điện năng tiêu giảm được 10,1% (năm 2016: 2.686.080 kW), một phần trong vấn đề giảm tiêu hao năng lượng là do có sự chuyển dịch về sản xuất sản phẩm sang 2 nhà máy Traphaco Hưng Yên và Traphaco CNC và áp dụng các biện pháp giảm tiêu hao năng lượng nhằm phát triển môi trường bền vững.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "25,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-e14c97c34811ab03-0-C3", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C3 direction", "claim": "Mức tiêu thụ điện năng của Công ty tăng 10,1% so với năm 2016.", "evidence": "So với năm 2016 mức tiêu thụ điện năng tiêu giảm được 10,1% (năm 2016: 2.686.080 kW), một phần trong vấn đề giảm tiêu hao năng lượng là do có sự chuyển dịch về sản xuất sản phẩm sang 2 nhà máy Traphaco Hưng Yên và Traphaco CNC và áp dụng các biện pháp giảm tiêu hao năng lượng nhằm phát triển môi trường bền vững.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-e14c97c34811ab03-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu mức tiêu thụ điện năng giảm 10,1% vào năm 2030.", "evidence": "So với năm 2016 mức tiêu thụ điện năng tiêu giảm được 10,1% (năm 2016: 2.686.080 kW), một phần trong vấn đề giảm tiêu hao năng lượng là do có sự chuyển dịch về sản xuất sản phẩm sang 2 nhà máy Traphaco Hưng Yên và Traphaco CNC và áp dụng các biện pháp giảm tiêu hao năng lượng nhằm phát triển môi trường bền vững.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-e14c97c34811ab03-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng điện từ năng lượng tái tạo của Công ty giảm 10,1% so với năm 2016.", "evidence": "So với năm 2016 mức tiêu thụ điện năng tiêu giảm được 10,1% (năm 2016: 2.686.080 kW), một phần trong vấn đề giảm tiêu hao năng lượng là do có sự chuyển dịch về sản xuất sản phẩm sang 2 nhà máy Traphaco Hưng Yên và Traphaco CNC và áp dụng các biện pháp giảm tiêu hao năng lượng nhằm phát triển môi trường bền vững.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng điện từ năng lượng tái tạo", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c385bf4d13ea40ed-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tỷ lệ rác thải được tái chế của Công ty đạt 75%.", "evidence": "»» Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy Tỷ lệ rác thải được tái chế chiếm 75% khối lượng rác thải của Công ty.", "labels": ["S"], "claim_ir": {"metric": "waste_recycled", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c385bf4d13ea40ed-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tỷ lệ rác thải được tái chế của Công ty đạt 90%.", "evidence": "»» Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy Tỷ lệ rác thải được tái chế chiếm 75% khối lượng rác thải của Công ty.", "labels": ["C"], "claim_ir": {"metric": "waste_recycled", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c385bf4d13ea40ed-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tỷ lệ rác thải được tái chế đạt 75% vào năm 2030.", "evidence": "»» Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy Tỷ lệ rác thải được tái chế chiếm 75% khối lượng rác thải của Công ty.", "labels": ["N"], "claim_ir": {"metric": "waste_recycled", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c385bf4d13ea40ed-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng chất thải phát sinh của Công ty đạt 75%.", "evidence": "»» Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy Tỷ lệ rác thải được tái chế chiếm 75% khối lượng rác thải của Công ty.", "labels": ["N"], "claim_ir": {"metric": "waste_generated", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng chất thải phát sinh", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-aa42b783af8c2d31-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Nước thải của Công ty đạt 100%.", "evidence": "100% nước thải từ sản xuất và sinh hoạt được xử lý đạt chuẩn trước khi xả thải.", "labels": ["S"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-aa42b783af8c2d31-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Nước thải của Công ty đạt 85%.", "evidence": "100% nước thải từ sản xuất và sinh hoạt được xử lý đạt chuẩn trước khi xả thải.", "labels": ["C"], "claim_ir": {"metric": "wastewater", "value_text": "85", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-aa42b783af8c2d31-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu nước thải đạt 100% vào năm 2030.", "evidence": "100% nước thải từ sản xuất và sinh hoạt được xử lý đạt chuẩn trước khi xả thải.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-aa42b783af8c2d31-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước sử dụng của Công ty đạt 100%.", "evidence": "100% nước thải từ sản xuất và sinh hoạt được xử lý đạt chuẩn trước khi xả thải.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-34d38386ad1c87b3-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Hệ thống năng lượng mặt trời của Công ty là 1,8 MWp.", "evidence": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện.", "labels": ["S"], "claim_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-34d38386ad1c87b3-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Hệ thống năng lượng mặt trời của Công ty là 2,2 MWp.", "evidence": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện.", "labels": ["C"], "claim_ir": {"metric": "renewable_energy", "value_text": "2,2", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-34d38386ad1c87b3-0-C2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C2 magnitude", "claim": "Hệ thống năng lượng mặt trời của Công ty là 1,8 nghìn MWp.", "evidence": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện.", "labels": ["C"], "claim_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-34d38386ad1c87b3-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu hệ thống năng lượng mặt trời đạt 1,8 MWp vào năm 2030.", "evidence": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-34d38386ad1c87b3-0-N5", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N5 basis", "claim": "Hệ thống năng lượng mặt trời của Công ty là 1,8 MWp/tấn sản phẩm.", "evidence": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-34d38386ad1c87b3-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Tổng lượng điện tiêu thụ của Công ty là 1,8 MWp.", "evidence": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng lượng điện tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Lượng nước trong hoạt động sản xuất của Công ty là 125.108 m3.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-S1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S1 rounded", "claim": "Lượng nước trong hoạt động sản xuất của Công ty là 125,1 nghìn m3.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "125,1", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Lượng nước trong hoạt động sản xuất của Công ty là 87.576 m3.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "87.576", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-C2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C2 magnitude", "claim": "Lượng nước trong hoạt động sản xuất của Công ty là 125.108 nghìn m3.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu Lượng nước trong hoạt động sản xuất đạt 125.108 m3 vào năm 2030.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-N5", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N5 basis", "claim": "Lượng nước trong hoạt động sản xuất của Công ty là 125.108 m3/tấn sản phẩm.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-N6", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N6 metric", "claim": "Lượng nước thải của Công ty là 125.108 m3.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ebf98e6bd3bc6405-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu của Công ty đạt 86,8%.", "evidence": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu: 86,8% Trụ sở chính, 11 Nhà Máy, 13 Trang trại được lắp đặt Hệ thống năng lượng mặt trời.", "labels": ["S"], "claim_ir": {"metric": "renewable_energy", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ebf98e6bd3bc6405-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu của Công ty đạt 71,8%.", "evidence": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu: 86,8% Trụ sở chính, 11 Nhà Máy, 13 Trang trại được lắp đặt Hệ thống năng lượng mặt trời.", "labels": ["C"], "claim_ir": {"metric": "renewable_energy", "value_text": "71,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ebf98e6bd3bc6405-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tỷ lệ năng lượng xanh, sạch/nhiên liệu đạt 86,8% vào năm 2030.", "evidence": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu: 86,8% Trụ sở chính, 11 Nhà Máy, 13 Trang trại được lắp đặt Hệ thống năng lượng mặt trời.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ebf98e6bd3bc6405-0-N6", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N6 metric", "claim": "Tổng lượng điện tiêu thụ của Công ty đạt 86,8%.", "evidence": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu: 86,8% Trụ sở chính, 11 Nhà Máy, 13 Trang trại được lắp đặt Hệ thống năng lượng mặt trời.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng lượng điện tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c46b97cef8117154-0-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Năm 2021, tỷ lệ tái sử dụng nước của Công ty đạt 90%.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-0-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Năm 2021, tỷ lệ tái sử dụng nước của Công ty đạt 75%.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-0-N1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N1 period", "claim": "Năm 2020, tỷ lệ tái sử dụng nước của Công ty đạt 90%.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2020, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-0-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu tỷ lệ tái sử dụng nước đạt 90% vào năm 2030.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-0-N6", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N6 metric", "claim": "Năm 2021, lượng nước thải của Công ty đạt 90%.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Năm 2021, lượng nước tái sử dụng của Công ty là 4000 m3.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Năm 2021, lượng nước tái sử dụng của Công ty là 3.200 m3.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "3.200", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-C2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C2 magnitude", "claim": "Năm 2021, lượng nước tái sử dụng của Công ty là 4000 nghìn m3.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-N1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N1 period", "claim": "Năm 2020, lượng nước tái sử dụng của Công ty là 4000 m3.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2020, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu lượng nước tái sử dụng đạt 4000 m3 vào năm 2030.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-N5", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N5 basis", "claim": "Năm 2021, lượng nước tái sử dụng của Công ty là 4000 m3/tấn sản phẩm.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-N6", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N6 metric", "claim": "Năm 2021, lượng nước thải của Công ty là 4000 m3.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-11bff39daa1d0a60-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Nước thải của Công ty đạt 100%.", "evidence": "100% Nước thải trong hoạt động Sản xuất và Chăn nuôi được xử lý đạt chuẩn.", "labels": ["S"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-11bff39daa1d0a60-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Nước thải của Công ty đạt 85%.", "evidence": "100% Nước thải trong hoạt động Sản xuất và Chăn nuôi được xử lý đạt chuẩn.", "labels": ["C"], "claim_ir": {"metric": "wastewater", "value_text": "85", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-11bff39daa1d0a60-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu Nước thải đạt 100% vào năm 2030.", "evidence": "100% Nước thải trong hoạt động Sản xuất và Chăn nuôi được xử lý đạt chuẩn.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-11bff39daa1d0a60-0-N6", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N6 metric", "claim": "Lượng nước sử dụng của Công ty đạt 100%.", "evidence": "100% Nước thải trong hoạt động Sản xuất và Chăn nuôi được xử lý đạt chuẩn.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Năm 2023, Tổng lượng nước tiêu thụ của Công ty là 253.145 m3.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-S1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S1 rounded", "claim": "Năm 2023, Tổng lượng nước tiêu thụ của Công ty là 253,1 nghìn m3.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "253,1", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Năm 2023, Tổng lượng nước tiêu thụ của Công ty là 354.403 m3.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "354.403", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-C2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C2 magnitude", "claim": "Năm 2023, Tổng lượng nước tiêu thụ của Công ty là 253.145 nghìn m3.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-N1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N1 period", "claim": "Năm 2022, Tổng lượng nước tiêu thụ của Công ty là 253.145 m3.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tổng lượng nước tiêu thụ đạt 253.145 m3 vào năm 2030.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-N5", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N5 basis", "claim": "Năm 2023, Tổng lượng nước tiêu thụ của Công ty là 253.145 m3/tấn sản phẩm.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Năm 2023, lượng nước thải của Công ty là 253.145 m3.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-5d74fc3fa628f4a0-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Nước thải của Công ty đạt 100%.", "evidence": "100% nước thải từ hoạt động sản xuất và sinh hoạt được xử lý đạt quy chuẩn trước khi xả thải vào nguồn tiếp nhận.", "labels": ["S"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-5d74fc3fa628f4a0-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Nước thải của Công ty đạt 85%.", "evidence": "100% nước thải từ hoạt động sản xuất và sinh hoạt được xử lý đạt quy chuẩn trước khi xả thải vào nguồn tiếp nhận.", "labels": ["C"], "claim_ir": {"metric": "wastewater", "value_text": "85", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-5d74fc3fa628f4a0-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu nước thải đạt 100% vào năm 2030.", "evidence": "100% nước thải từ hoạt động sản xuất và sinh hoạt được xử lý đạt quy chuẩn trước khi xả thải vào nguồn tiếp nhận.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-5d74fc3fa628f4a0-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước sử dụng của Công ty đạt 100%.", "evidence": "100% nước thải từ hoạt động sản xuất và sinh hoạt được xử lý đạt quy chuẩn trước khi xả thải vào nguồn tiếp nhận.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8a12299d98291537-0-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Tiêu thụ gas của Công ty giảm 1,4% so với năm 2021.", "evidence": "Trong bối cảnh năng suất tăng hơn 60%, áp lực đảm bảo chỉ tiêu cường độ tiêu hao năng lượng là Dù chưa được như kỳ vọng nhưng mức giảm tiêu thụ gas 1,4% so với năm 2021 là kết quả đáng ghi nhận.", "labels": ["S"], "claim_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ gas", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức giảm tiêu thụ gas", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-8a12299d98291537-0-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Tiêu thụ gas của Công ty giảm 16,4% so với năm 2021.", "evidence": "Trong bối cảnh năng suất tăng hơn 60%, áp lực đảm bảo chỉ tiêu cường độ tiêu hao năng lượng là Dù chưa được như kỳ vọng nhưng mức giảm tiêu thụ gas 1,4% so với năm 2021 là kết quả đáng ghi nhận.", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "16,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ gas", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức giảm tiêu thụ gas", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-8a12299d98291537-0-C3", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C3 direction", "claim": "Tiêu thụ gas của Công ty tăng 1,4% so với năm 2021.", "evidence": "Trong bối cảnh năng suất tăng hơn 60%, áp lực đảm bảo chỉ tiêu cường độ tiêu hao năng lượng là Dù chưa được như kỳ vọng nhưng mức giảm tiêu thụ gas 1,4% so với năm 2021 là kết quả đáng ghi nhận.", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ gas", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức giảm tiêu thụ gas", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-8a12299d98291537-0-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu tiêu thụ gas giảm 1,4% vào năm 2030.", "evidence": "Trong bối cảnh năng suất tăng hơn 60%, áp lực đảm bảo chỉ tiêu cường độ tiêu hao năng lượng là Dù chưa được như kỳ vọng nhưng mức giảm tiêu thụ gas 1,4% so với năm 2021 là kết quả đáng ghi nhận.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ gas", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức giảm tiêu thụ gas", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-d08482796aa8de43-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Nước thải của Công ty đạt 100%.", "evidence": "Không có khiếu nại của cộng đồng xung quanh về Không có khiếu nại về bất bình đẳng Không có khiếu nại về phân biệt đối xử Không lao động cưỡng bức, không lao động trẻ em 100% Nước thải hoạt động sản xuất được xử lý trước khi thải ra môi trường.", "labels": ["S"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-d08482796aa8de43-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Nước thải của Công ty đạt 85%.", "evidence": "Không có khiếu nại của cộng đồng xung quanh về Không có khiếu nại về bất bình đẳng Không có khiếu nại về phân biệt đối xử Không lao động cưỡng bức, không lao động trẻ em 100% Nước thải hoạt động sản xuất được xử lý trước khi thải ra môi trường.", "labels": ["C"], "claim_ir": {"metric": "wastewater", "value_text": "85", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-d08482796aa8de43-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu Nước thải đạt 100% vào năm 2030.", "evidence": "Không có khiếu nại của cộng đồng xung quanh về Không có khiếu nại về bất bình đẳng Không có khiếu nại về phân biệt đối xử Không lao động cưỡng bức, không lao động trẻ em 100% Nước thải hoạt động sản xuất được xử lý trước khi thải ra môi trường.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-d08482796aa8de43-0-N6", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N6 metric", "claim": "Lượng nước sử dụng của Công ty đạt 100%.", "evidence": "Không có khiếu nại của cộng đồng xung quanh về Không có khiếu nại về bất bình đẳng Không có khiếu nại về phân biệt đối xử Không lao động cưỡng bức, không lao động trẻ em 100% Nước thải hoạt động sản xuất được xử lý trước khi thải ra môi trường.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Năng lượng mặt trời của Công ty là 14.421.569 Kwh.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["S"], "claim_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-S1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S1 rounded", "claim": "Năng lượng mặt trời của Công ty là 14,4 triệu Kwh.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["S"], "claim_ir": {"metric": "renewable_energy", "value_text": "14,4", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Năng lượng mặt trời của Công ty là 18.026.961 Kwh.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["C"], "claim_ir": {"metric": "renewable_energy", "value_text": "18.026.961", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-C2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C2 magnitude", "claim": "Năng lượng mặt trời của Công ty là 14.421.569 nghìn Kwh.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["C"], "claim_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu năng lượng mặt trời đạt 14.421.569 Kwh vào năm 2030.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-N5", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N5 basis", "claim": "Năng lượng mặt trời của Công ty là 14.421.569 Kwh/tấn sản phẩm.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-N6", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N6 metric", "claim": "Tổng lượng điện tiêu thụ của Công ty là 14.421.569 Kwh.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng lượng điện tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n',
    'candidates.jsonl': '{"id": "f386b5bf3d338069", "company_id": "bvh", "reporting_year": 2025, "text": "Biện pháp xử lý kỹ thuật Giải pháp cộng đồng & Truyền thông nội bộ 146kg Bóng đèn + 31kg Pin được thu gom, tái chế thay vì thải ra môi trường tương đương giảm phát thải"}\n{"id": "8fd05beb5922e2aa", "company_id": "bvh", "reporting_year": 2025, "text": "mục tiêu 963 tấn CO2 theo kịch bản lộ trình, chênh lệch này chủ yếu xuất phát từ các yếu tố khách quan như gia tăng hoạt động kinh doanh ngắn hạn, nhu cầu sửa chữa và cải tạo tòa nhà, mở rộng quy mô vận hành, cũng như biến động nhu cầu sử dụng năng lượng trong bối cảnh thời tiết cực đoan do biến đổi khí hậu."}\n{"id": "5f1469b011aaf199", "company_id": "tra", "reporting_year": null, "text": "năm đầu tiên Công ty triển khai Chiến lược phát triển bền vững Traphaco giai này, nhóm cổ đông lớn chiếm 40% vốn điều lệ, thực hiện quá trình thoái vốn tại Traphaco;"}\n{"id": "1bd4e7d8ed9cc750", "company_id": "pnj", "reporting_year": null, "text": "thụ năng lượng điện phục vụ cho các hoạt động của mình, PNJP còn có một máy phát điện dự phòng với công suất 870KVA để sử dụng trong trường hợp mất điện, nhiên liệu dùng cho máy phát điện là 120 lít dầu DO dùng cho 1 giờ mất điện."}\n{"id": "6ee07e1fefd815d0", "company_id": "dhg", "reporting_year": 2024, "text": "Nước thải sinh hoạt của CBNV trung bình 1.531 m3/tháng."}\n{"id": "bf08a85bc035d56c", "company_id": "tra", "reporting_year": null, "text": "Tuần hoàn nguồn nước từ hệ thống RO để làm mát thiết bị với khối lượng ước tính tái sử dụng là 1.560 m3."}\n{"id": "9614b932cde64c83", "company_id": "tra", "reporting_year": null, "text": "Xử lý nước thải công nghiệp bằng công nghệ xử lý hóa lý kết hợp sinh học hiếu khí và công nghệ Chất lượng nước đầu ra Đạt QCVN40:2011/BTNMT (cột A) - Tương ứng với nước dùng được Công suất: 60m3/ ngày đêm."}\n{"id": "7d89765a71e6e33d", "company_id": "dhg", "reporting_year": 2023, "text": "vừa thể hiện cơ hội, tiềm năng nhưng cũng là thách thức không nhỏ cho các doanh nghiệp dược trong nước để phấn đấu đưa giá trị thuốc sản xuất trong nước chiếm 70% thị phần theo chủ trương của Chính Phủ."}\n{"id": "2a31912f6d377a83", "company_id": "tra", "reporting_year": null, "text": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3."}\n{"id": "2c4202d9049e84ca", "company_id": "dhg", "reporting_year": 2024, "text": "Nguồn: BCTC hợp nhất đã kiểm toán của DHG Pharma Nguồn: BCTC hợp nhất đã kiểm toán của DHG Pharma DOANH THU THUẦN (Tỷ đồng) Kế hoạch Thực tế Hàng trong nước Xuất khẩu Khác LỢI NHUẬN TRƯỚC THUẾ (Tỷ đồng) Kế hoạch Thực tế Tổng doanh thu thuần năm 2022 đạt tăng 16,8% so với cùng kỳ DHG Pharma | Báo cáo phát triển bền vững 2022 Các lĩnh vực trọng yếu"}\n{"id": "af7a8b8ca7f93b26", "company_id": "ctd", "reporting_year": 2025, "text": "52,02% tổng diện tích của dự án đã được phủ xanh bằng các loài cây bản địa chịu hạn, góp phần giảm thiểu tác động đến tài nguyên nước và hỗ trợ bảo vệ môi trường tự nhiên."}\n{"id": "69add0ecad62238e", "company_id": "pnj", "reporting_year": 2021, "text": "- Xây dựng hệ thống xử lý chất thải tiên tiến, xử lý tối đa lượng chất thải trước khi đưa ra môi trường và liên kết bên thứ ba để sử lý 100% lượng phát thải."}\n{"id": "f5aa52feaf9057eb", "company_id": "tra", "reporting_year": null, "text": "dược liệu nguồn gốc trong nước chiếm 91,3% tổng nhu cầu dược liệu sử dụng sản xuất với các vùng"}\n{"id": "f5f8a5c13872aea7", "company_id": "tra", "reporting_year": 2025, "text": "Mức tiêu thụ nước giảm : 12,8% (so với năm 2024: 4.878 m3) VÀ PHÁT THẢI HIỆU QUẢ • Phun nước làm ẩm đường vào những ngày khô hanh."}\n{"id": "819e251cae778a40", "company_id": "dhg", "reporting_year": 2025, "text": "Với tầm nhìn dài hạn, Chính phủ đặt mục tiêu GDP năm 2025 tăng trưởng ít nhất 8%, vượt xa dự báo 6,1% - 6,7% từ các tổ chức quốc tế, thể hiện quyết tâm mạnh mẽ trong việc thúc đẩy phát triển kinh tế bền vững."}\n{"id": "8a824e87384adb58", "company_id": "pow", "reporting_year": 2025, "text": "bằng 84% kế hoạch, tăng Đạt 1.684 tỷ đồng, đạt Nộp Ngân sách Nhà nước: đồng, bằng 227% kế hoạch."}\n{"id": "af2808ae365c48da", "company_id": "bvh", "reporting_year": 2025, "text": "Tuy nhiên, mức tăng dự kiến không quá lớn và vẫn duy trì trong vùng hỗ trợ cho hoạt động sản xuất kinh doanh của Ngân hàng Nhà nước (NHNN) Việt Nam dự kiến mục tiêu tăng trưởng tín dụng toàn hệ thống năm 2026 ở mức khoảng 15%, thấp hơn so với mức thực tế năm 2025 (19,1%) nhưng phù hợp với định hướng ổn định vĩ mô."}\n{"id": "27a13a6adcc4f0b6", "company_id": "tra", "reporting_year": 2025, "text": "DOANH THU HỢP NHẤT NĂM 2024 Tăng trưởng 1,2% so với năm 2023 ĐÓNG GÓP CHO NGÂN SÁCH NHÀ NƯỚC CHI TRẢ CỔ TỨC CHO CỔ ĐÔNG LỢI NHUẬN SAU THUẾ NĂM 2024 Traphaco đạt tỉ lệ tăng trưởng về doanh thu và lợi nhuận lần lượt là 5,19% và 5,04% Báo cáo thường niên 2024 BÁO CÁO PHÁT TRIỂN BỀN VỮNG"}\n{"id": "56751234aa2257b5", "company_id": "tra", "reporting_year": null, "text": "Sử dụng nước thải sau xử lý đạt tiêu chuẩn để phục vụ mục đích tưới cây và vệ sinh khuôn viên: khoảng 2% lượng nước đầu vào."}\n{"id": "9c78bf02decaf05d", "company_id": "pnj", "reporting_year": 2023, "text": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm"}\n{"id": "010ce3c27e3667c8", "company_id": "ctd", "reporting_year": 2025, "text": "Song song, 26% dòng vật liệu bền Những kết quả này là minh chứng cho cam kết của Coteccons trong việc tiên phong phát triển bền vững và xây dựng những công trình xanh, thân thiện với môi trường."}\n{"id": "943beeaef7a587ea", "company_id": "tra", "reporting_year": null, "text": "Chất thải rắn thông thường năm Rác thải sinh hoạt 9.600 kg/năm."}\n{"id": "270002ea10dc9ad7", "company_id": "tra", "reporting_year": 2025, "text": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp."}\n{"id": "9796bc7d982bb3ce", "company_id": "tra", "reporting_year": null, "text": "TIÊU THỤ TÀI NGUYÊN ĐIỆN, NƯỚC, GIẤY Năm 2014, tổng tiêu thụ điện khu vực văn phòng giảm 6% so với năm 2013, duy trì tiết kiệm điện nước, ổn định mức tiêu thụ điện nước cho sản xuất, lượng giấy sử dụng cho toàn bộ hoạt động của Công ty."}\n{"id": "c5b8455b08f5a561", "company_id": "tra", "reporting_year": null, "text": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;"}\n{"id": "00e3118d08d4c8b7", "company_id": "bvh", "reporting_year": 2025, "text": "sẽ tiếp tục tìm kiếm các giải pháp sáng tạo và bền vững để duy trì mức phát thải dưới 900 tấn CO2 vào năm 2026."}\n{"id": "d7a04b919bbe8cda", "company_id": "bvh", "reporting_year": 2025, "text": "nước sinh hoạt chiếm tỷ trọng lớn, khoảng 4.849 m3, trong khi nước uống tinh khiết So với năm 2024, lượng nước tiêu thụ ghi nhận xu hướng giảm đáng kể khoảng 41%, cho thấy hiệu quả của các giải pháp kiểm soát và tối ưu sử dụng nước trong toàn hệ thống."}\n{"id": "49415c95a77767b0", "company_id": "tra", "reporting_year": null, "text": "Nước thải và Chất thải toàn bộ đều qua hệ thống xử lý và đảm bảo an toàn 100% với môi trường thông qua các đơn vị xử lý trước khi đến địa điểm xả thải."}\n{"id": "cfbc67bcf17fd391", "company_id": "bvh", "reporting_year": 2025, "text": "quan BHXH Việt Nam dẫn nguồn Tổng cục Thống kê, quý IV/2025 cả nước có khoảng 32,7 triệu lao động phi chính thức, chiếm 61,9% tổng số lao động có việc làm."}\n{"id": "4c3c4794cb3161b7", "company_id": "ctd", "reporting_year": 2025, "text": "với việc ưu tiên sử dụng vật liệu gạch không nung cũng góp phần tiết giảm được 926 tấn CO2, tiết kiệm được 171 tấn dầu tương đương."}\n{"id": "23f7a02fff86669c", "company_id": "pow", "reporting_year": 2025, "text": "KG/ Siemes Energy Limited Company ký ngày 28 tháng 7 năm 2023."}\n{"id": "c548ddc6321f99a1", "company_id": "dhg", "reporting_year": 2025, "text": "trên tổng tài sản vẫn ở mức an toàn (10,9%) và tỷ lệ đòn bẩy tài chính (D/E) tăng nhẹ từ 0,12 lên 0,16 lần – vẫn trong vùng kiểm soát tốt, phản ánh cơ cấu tài chính lành mạnh và chính sách sử dụng vốn thận trọng."}\n{"id": "6bd49760519ee670", "company_id": "pow", "reporting_year": 2025, "text": "NT2 được miễn thuế 04 năm kể từ khi có thu nhập chịu thuế đối với hoạt động sản xuất kinh doanh điện năng và giảm 50% số thuế phải nộp cho 09 năm tiếp theo."}\n{"id": "648b1dc6867dc23c", "company_id": "dhg", "reporting_year": 2024, "text": "Cả 2 kênh đều có mức tăng trưởng mạnh so với cùng kỳ, trong đó Kênh Pharmacy đóng góp 77% trong tổng mức tăng trưởng 562 tỷ đồng của doanh thu hàng trong nước."}\n{"id": "b13683e8713dc075", "company_id": "tra", "reporting_year": null, "text": "Hiện nay, TRAPHACO đang chiếm khoảng 4% thị phần thuốc sản xuất trong nước."}\n{"id": "f50782a48df3170e", "company_id": "tra", "reporting_year": 2025, "text": "Giai đoạn 2 (từ 6/7/2021 đến 6/10/2021): Do ảnh hưởng đáng kể của đợt bùng phát dịch Covid-19 lần thứ tư trên cả nước, đặc biệt tại khu vực miền Nam, VN-Index đã sụt giảm 31,81% xuống gần sát vùng giá đầu năm khi chạm đáy 1.220 điểm vào ngày 19/07/2021."}\n{"id": "5132e7d323883421", "company_id": "tra", "reporting_year": null, "text": "20 bệnh phổ biến nhất tại các nước phát triển tiêu tốn Thuốc điều trị viêm gan siêu vi sẽ tăng trưởng gần 38%, trong khi thuốc tim mạch, mỡ máu, kháng sinh, kháng viêm..."}\n{"id": "f2d7d27ebef7e5f0", "company_id": "tra", "reporting_year": null, "text": "Sản xuất liên tục để giảm số lần vệ sinh máy, giảm têu hao năng lượng nước, giảm nguồn nước thải Sử dụng nước thải sau xử lý đạt têu chuẩn để phục vụ mục đích tưới cây và vệ sinh khuôn viên: khoảng 2% lượng nước đầu vào."}\n{"id": "f996295e3a2f0cae", "company_id": "pnj", "reporting_year": 2021, "text": "Vòng quay hàng tồn kho tăng gần gấp đôi so với CÔNG TY TNHH MTV GIÁM ĐỊNH PNJ (PNJL) PNJL khẳng định là đơn vị giám định kim cương, đá quý, kim loại quý hàng đầu thị trường Việt Nam hiện nay và là một trong số ít công ty kiểm định trong nước xây dựng, vận hành thành công theo tiêu chuẩn Hàng hóa được giao đúng hẹn với tỉ lệ trên 90%."}\n{"id": "3bffc56b2f545687", "company_id": "tra", "reporting_year": 2025, "text": "MÔI TRƯỜNG (Tiếp theo) KIỂM SOÁT NƯỚC THẢI VÀ CHẤT THẢI Hệ thống xử lý nước thải: • Hệ thống thu gom, xử lý nước thải được xây dựng theo công nghệ sinh học, công suất thiết kế 30 m3/ngày đêm."}\n{"id": "16465492a67586d8", "company_id": "tra", "reporting_year": 2024, "text": "Một số sản phẩm chính mang lại doanh thu chủ yếu cho Công ty như: Dung dịch nhỏ mắt, mũi Natri Clorid 0,9%, Thuốc ho Methorphan, Dibetalic, nước súc miệng T-B, viên nén bao phim Azizi..."}\n{"id": "34607b5d58a38b25", "company_id": "pnj", "reporting_year": 2021, "text": "Lưu lượng nước thải phát sinh từ quá trình sinh hoạt của Công ty khoảng 80~9m3/ngày Thành phần nước thải này chứa chủ yếu là: các chất cặn bã, các chất dinh dưỡng (N, P), chất rắn lơ lửng (SS), COD, các chất hữu cơ (BOD5 và các vi khuẩn)..."}\n{"id": "db8841dc88c6b9be", "company_id": "tra", "reporting_year": 2023, "text": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội."}\n{"id": "3314c38da596331a", "company_id": "bvh", "reporting_year": 2025, "text": "Đầu tư công tiếp tục đóng vai trò quan trọng trong việc hỗ trợ tăng trưởng Tổng vốn đầu tư toàn xã hội năm 2025 tăng khoảng 11,6%, trong đó vốn đầu tư từ ngân sách nhà nước tăng gần 28% so với năm trước."}\n{"id": "a201f04f3bd17773", "company_id": "tra", "reporting_year": 2023, "text": "đông dược, chúng ta chứng kiến mức tăng trưởng +37% so với năm 2021, đồng thời mang lại mức tăng trưởng +16% cả đông dược và ngoài thảo dược Chúng ta sẽ tiếp tục nỗ lực mở rộng hơn nữa chiến lược này sang các địa bàn khác trên cả nước trong năm 2023, qua đó đạt được sự tăng trưởng vững chắc ở cả hai mảng đông dược và ngoài đông dược."}\n{"id": "f6bbb222669b318a", "company_id": "bvh", "reporting_year": 2025, "text": "Các nghiên cứu dự báo cho thấy nếu nhiệt độ toàn cầu tiếp tục gia tăng, mực nước biển tại khu vực Đồng bằng sông Cửu Long có thể tăng từ 55 - 75 cm trong thế kỷ tới, khiến khoảng 40% diện tích khu vực này có nguy cơ bị ngập, đồng thời làm gia tăng tình trạng xâm nhập mặn và ảnh hưởng đến sinh kế của người dân."}\n{"id": "61065addf3e8f32f", "company_id": "cdn", "reporting_year": 2025, "text": "Ban Kiểm soát báo cáo Đại hội đồng cổ đông: - Theo quy định của Luật Chứng khoán và ý kiến của Ủy ban Chứng khoán Nhà nước, tại thời điểm 01/01/2026, Công ty Cổ phần Cảng Đà Nẵng không còn đáp ứng điều kiện về cơ cấu cổ đông của công ty đại chúng (tỷ lệ cổ phần do cổ đông không phải cổ đông lớn nắm giữ dưới 10%)."}\n{"id": "dd41a402d866f65e", "company_id": "pnj", "reporting_year": null, "text": "Với tốc độ tăng nhu cầu năng lượng bình quân 5,2%/năm, việc sử dụng năng lượng tiết kiệm và hiệu quả có ý nghĩa rất quan trọng trong quá trình phát triển năng lượng và đảm bảo an ninh năng lượng quốc gia."}\n{"id": "6e2a279573ee4ff1", "company_id": "vnm", "reporting_year": 2022, "text": "4,99% Lượng nước trong hoạt động sản xuất được tận thu tuần hoàn và tái sử dụng."}\n{"id": "d2c1ce4ffefdc181", "company_id": "pnj", "reporting_year": 2021, "text": "nước thải sản xuất phát sinh khoảng 50~60 m3/ngày x đêm."}\n{"id": "6438fbfc923aa6ea", "company_id": "tra", "reporting_year": null, "text": "Công nghệ hệ thống xử lý nước thải »» Hệ thống thu gom, xử lý nước thải công nghiệp được xây dựng theo công nghệ sinh học, công suất thiết kế 30 m3/ngày đêm."}\n{"id": "7d2843caa6000cf5", "company_id": "pow", "reporting_year": 2025, "text": "Đối với Quỹ phát triển khoa học và công nghệ không được sử dụng hết 70% trong vòng 5 năm kể từ khi trích lập quỹ, Tổng công ty có nghĩa vụ nộp Ngân sách Nhà nước phần thuế thu nhập doanh nghiệp tính trên số tiền quỹ còn lại và phần lãi trả chậm phát sinh từ số thuế thu nhập doanh nghiệp đó."}\n{"id": "4a447cacf15eaee2", "company_id": "dhg", "reporting_year": 2024, "text": "Nước thải sinh hoạt được xử lý sơ bộ qua hầm tự hoại 03 ngăn, sau đó được thu gom về 02 hệ thống xử lý tập trung với công suất 400 m3/ngày đêm/hệ thống để xử lý cùng với nước thải sản xuất."}\n{"id": "7843554e8927d787", "company_id": "dhg", "reporting_year": 2024, "text": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí)."}\n{"id": "134a1fa5af75e22a", "company_id": "pnj", "reporting_year": 2023, "text": "Năm 2023, PNJ đặt mục tiêu giảm năng lượng theo lộ trình giảm phát thải ròng bằng “Không” theo mục tiêu quốc gia (mỗi năm giảm 1%), không ngừng chuẩn hóa thiết bị đo đạc để xác định rõ dòng năng lượng tiêu thụ."}\n{"id": "29965813597a3136", "company_id": "dhg", "reporting_year": 2024, "text": "Thuế giá trị gia tăng (GTGT) chiếm tỷ trọng cao nhất, gần 50% trong cơ cấu Kế đến là thuế thu nhập doanh nghiệp (TNDN) và thuế thu nhập cá Tổng giá trị nộp ngân sách nhà nước trong 5 năm gần nhất là Tại DHG Pharma, Công ty cũng sở hữu Nhà máy Dược phẩm DHG và Nhà máy In Bao bì tại Hậu Giang thuộc diện ưu đãi."}\n{"id": "b89219ea98c4b6f9", "company_id": "ctd", "reporting_year": 2025, "text": "Lợi nhuận trước thuế Theo báo cáo của Bộ Xây dựng, tốc độ tăng trưởng của ngành xây dựng trong nước đạt 7,34% trong 6 tháng đầu Tốc độ tăng trưởng của hoạt động kinh doanh bất động sản tăng 2,45% so với cùng kỳ năm ngoái."}\n{"id": "2995ad77ed6c947e", "company_id": "tra", "reporting_year": 2023, "text": "Tuân thủ các quy định liên quan đến bảo tồn đa dạng sinh học, phát triển bền vững và chia sẻ công bằng lợi ích giữa các bên liên quan, hướng tới phù hợp với các tiêu chuẩn của Công ước Đảm bảo 100% các đơn hàng được giao đúng cam kết."}\n{"id": "eac4d4d9ee03aae3", "company_id": "vnm", "reporting_year": 2022, "text": "Nhờ những hành động trên, Vinamilk là doanh nghiệp đầu tiên tại Châu Á vinh dự đón nhận giải thưởng Purity Award của Hoa Kỳ cho sữa bột trẻ em Optimum Gold và Clean Label Project chứng nhận sạch cho sữa tươi 100% Green Farm và Organic, là minh chứng rõ ràng cho chất lượng đạt chuẩn quốc tế của các sản phẩm của Vinamilk."}\n{"id": "ff5b1358bf5a350d", "company_id": "tra", "reporting_year": 2023, "text": "KIỂM SOÁT NƯỚC THẢI VÀ CHẤT THẢI • Hệ thống xử lý nước thải: Hệ thống thu gom, xử lý nước thải được xây dựng theo công nghệ sinh học, công suất thiết kế 30 m3/ngày đêm."}\n{"id": "bcb71c95fe65b777", "company_id": "dhg", "reporting_year": 2024, "text": "Lượng CO2 cắt giảm (từ năm thứ nhất đến năm thứ 20): khoảng 53 tấn."}\n{"id": "16bf11595e8fc1a7", "company_id": "ctd", "reporting_year": 2025, "text": "ước tính, một dự án như vậy khi hoàn thành sẽ loại trừ và giảm tổng cộng 2.803 tấn carbon hàm chứa so với vật liệu thông thường, tương đương với lượng carbon được hấp thụ bởi 46.348 cây xanh trong 10 năm."}\n{"id": "68c5b15f5e0577aa", "company_id": "pow", "reporting_year": 2025, "text": "Các chỉ tiêu chính cả năm 2025 đạt được như sau: Năm 2025, sản lượng điện Doanh thu cả năm 2025 Lợi nhuận trước thuế đạt 20,7 tỷ đồng Nộp Ngân sách nhà nước đạt 0,87 tỷ đồng Năm 2025, sản lượng điện đạt 5,1 triệu kWh TÌNH HÌNH HOẠT ĐỘNG NĂM 2025 TỔNG CÔNG TY ĐIỆN LỰC DẦU KHÍ VIỆT NAM - CTCP KHỞI ĐẦU MỚI - THÀNH TỰU MỚI"}\n{"id": "aa1899e93390435a", "company_id": "vnm", "reporting_year": 2023, "text": "Công ty nhận được Giấy Chứng nhận Đăng ký Đầu tư ra nước ngoài số 201600140, đăng ký điều chỉnh lần thứ 2 về việc tăng vốn đầu tư của Công ty tại công ty con do Công ty sở hữu 100% là Driftwood Dairy Holding Corporation, bang California, Hoa Kỳ, từ 10 triệu USD lên 20 triệu USD."}\n{"id": "5bbde4a73faecf84", "company_id": "sab", "reporting_year": null, "text": "Xuất khẩu: Bên cạnh thị trường trong nước, trong năm 2018, SABECO gia tăng về sản lượng xuất khẩu, khi sản lượng bia xuất khẩu đạt 33 triệu lít, chiếm 1,8% tổng sản lượng tiêu thụ bia của SABECO, trong đó khu vực Tây Phi có tốc độ và dư địa tăng trưởng cao nhất."}\n{"id": "8f76ce4f578023ec", "company_id": "pow", "reporting_year": 2025, "text": "và hệ số lợi nhuận sau thuế trên tổng tài sản bình quân tăng từ 1,61% lên 3,57%, phản ánh hiệu quả sử dụng vốn và tài sản được nâng cao rõ rệt."}\n{"id": "f57cad454e30e881", "company_id": "dhg", "reporting_year": 2025, "text": "Đặc biệt, DHG Pharma đã đầu tư hệ thống điện mặt trời áp mái tại khu vực nhà xưởng và công trình phụ trợ như Xưởng 1, Xưởng Bao bì, Kho Thành phẩm 2 (giai đoạn 1), Nhà ăn – Hội trường với tổng công suất lắp đặt 1,8 MWp."}\n{"id": "f73f832724654622", "company_id": "bvh", "reporting_year": 2025, "text": "tiềm năng tiết kiệm năng lượng trong các tòa nhà đến từ các thiết bị sẽ tạo ra tính tối ưu trong sử dụng năng lượng, chi phí vận hành, dự kiến khoảng 10 - 40% chi phí."}\n{"id": "2459d72eecc55f75", "company_id": "pow", "reporting_year": 2025, "text": "Vốn điều lệ: 270.000.000.000 đồng Tỷ lệ nắm giữ của PV Power: 51% CTCP Năng lượng tái tạo Điện lực Dầu khí TỔNG CÔNG TY ĐIỆN LỰC DẦU KHÍ VIỆT NAM - CTCP KHỞI ĐẦU MỚI - THÀNH TỰU MỚI"}\n{"id": "9796cd11c4d01aea", "company_id": "ctd", "reporting_year": 2025, "text": "Theo ước lượng, ngành xây dựng sử dụng đến 40% tổng năng lượng toàn cầu, chịu trách nhiệm cho hơn 30% tổng lượng phát thải khí nhà kính và liên quan đến lượng lao động rất lớn."}\n{"id": "ebd86f76835f1544", "company_id": "tra", "reporting_year": null, "text": "Năm 2017, Việt Nam nhập khẩu 375 triệu USD nguyên phụ liệu dược phẩm, 78% trong số đó là từ Trung Quốc và Ấn Độ - là hai nguồn cung cấp thuốc chủ yếu cho các nước đang phát triển có tham gia sản xuất thuốc generic."}\n{"id": "a1b9ae421da67dd8", "company_id": "dhg", "reporting_year": 2025, "text": "9,66 tấn, với chi phí khoảng 218 triệu đồng, chủ yếu phục vụ hoạt động sản xuất tại xưởng thuốc nước tại Cần Thơ."}\n{"id": "ee4b9253a749cd8a", "company_id": "dhg", "reporting_year": 2025, "text": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường."}\n{"id": "631428f0da19badc", "company_id": "vnm", "reporting_year": 2022, "text": "Tỷ lệ sử dụng năng lượng xanh và sạch trong hoạt động sản xuất đạt 87%."}\n{"id": "e14c97c34811ab03", "company_id": "tra", "reporting_year": null, "text": "So với năm 2016 mức tiêu thụ điện năng tiêu giảm được 10,1% (năm 2016: 2.686.080 kW), một phần trong vấn đề giảm tiêu hao năng lượng là do có sự chuyển dịch về sản xuất sản phẩm sang 2 nhà máy Traphaco Hưng Yên và Traphaco CNC và áp dụng các biện pháp giảm tiêu hao năng lượng nhằm phát triển môi trường bền vững."}\n{"id": "e30dab72e42aed25", "company_id": "tra", "reporting_year": null, "text": "Nợ phải trả và nguồn vốn chủ sở hữu trên tổng nguồn vốn giảm nhẹ so với cùng kỳ năm 2018, trong đó cơ cấu nguồn vốn chủ sở hữu trên tổng nguồn vốn đạt 65,64% thể hiện cơ cấu tài chính Công ty có tính minh bạch và bền vững."}\n{"id": "53cd2b44a596cb43", "company_id": "tra", "reporting_year": 2024, "text": "Hệ thống xử lý nước thải: » Hệ thống thu gom, xử lý nước thải được xây dựng theo công nghệ sinh học, công suất thiết kế 30 m3/ngày đêm."}\n{"id": "12e6ba96428e267a", "company_id": "bvh", "reporting_year": 2025, "text": "50% tòa nhà lắp đặt hệ thống năng lượng tái tạo và giảm thiểu sử dụng năng lượng không tái • Khuyến khích sử dụng văn phòng phẩm làm từ vật liệu tái chế, thân thiện với môi trường • Tạo không gian xanh tại các văn phòng và trồng cây xanh."}\n{"id": "58931ceefdd17d3e", "company_id": "dhg", "reporting_year": 2024, "text": "Năm bản lề Dược Hậu Giang thực hiện tái cấu trúc, củng cố lại mọi hoạt động của Công ty để chuẩn bị cho một chu kỳ tăng trưởng mới Điều chỉnh giới hạn tỷ lệ sở hữu của nhà đầu tư nước ngoài tại DHG Pharma từ 49% lên 100% vốn điều lệ."}\n{"id": "b3648e71b99cea72", "company_id": "pan", "reporting_year": null, "text": "SUSTAINABILITY REPORT 19 OVERVIEW ON SUSTAINABILITY REPORT THÔNG TIN CHUNG VỀ TẬP ĐOÀN THE PAN GROUP OVERVIEW was established with distributor of Nilfisk Services Ha Noi, in- vices Hanoi to 100%."}\n{"id": "6ca796dacbfd4116", "company_id": "pnj", "reporting_year": null, "text": "Lưu lượng nước thải phát sinh từ quá trình sinh hoạt của Công ty khoảng 80~90 m3/ Thành phần nước thải này chứa chủ yếu là: các chất cặn bã, các chất dinh dưỡng (N, P), chất rắn lơ lửng (SS), COD, các chất hữu cơ (BOD5 và các vi khuẩn)..."}\n{"id": "39e9ef4be71ad5b8", "company_id": "dhg", "reporting_year": 2024, "text": "Nhờ đó, doanh thu thuần của Kênh Hospital tăng 33% so với cùng kỳ và chiếm tỷ trọng khoảng 13% trong tổng doanh thu hàng trong nước."}\n{"id": "765341cf7fbf2a73", "company_id": "tra", "reporting_year": null, "text": "Trong khi đó, lao động dưới 30 tuổi chiếm tỉ lệ thôi việc là 1,37% và lao động trong độ tuổi từ 30 - 50 tuổi có tỉ lệ thôi việc cao nhất Với đội ngũ lao động ngày càng gia tăng về chất lượng đã đáp ứng được tốt các yêu cầu kinh doanh và phát triển bền vững của Traphaco."}\n{"id": "80e6c83c3d8e5b63", "company_id": "tra", "reporting_year": 2023, "text": "Nợ phải trả và nguồn vốn chủ sở hữu trên tổng nguồn vốncó sự điều chỉnh nhẹ trong năm 2022, trong đó cơ cấu nguồn vốn chủ sở hữu trên tổng nguồn vốn đạt 76,01% , tăng 1,76% trong khi nợ phải trả giảm gần 2 điểm phần trăm cho thấy cơ cấu tài chính công ty có tính minh bạch và bền vững."}\n{"id": "0273e8342476d770", "company_id": "tra", "reporting_year": null, "text": "Các nhà máy được xây dựng mới dành 20% kinh phí đầu tư cho việc xử lý môi trường, cập nhật công nghệ sản xuất hiện đại cùng với việc áp dụng hệ thống quản trị tiên tiến, giảm thiểu tác động môi trường, và tạo ra năng suất lao động cao, hiệu quả kinh doanh từ tiết kiệm năng lượng và giảm thiểu nguồn rác thải."}\n{"id": "5ef27103e1bc610e", "company_id": "ctd", "reporting_year": 2025, "text": "đốc dự án, cần tập trung đào tạo về kỹ năng đàm phán với khách hàng, quản lý dự án, • Tăng cường tìm kiếm và đào tạo nhân sự làm việc tại nước ngoài, nhân sự phục vụ cho quá trình đồng vận hành (synergy) tại những công Cải thiện tỷ lệ lợi nhuận Tỷ lệ lợi nhuận gộp (3,39%) và tỷ lệ lợi nhuận ròng (1,84%) năm 2024 đều tăng so với năm 2023."}\n{"id": "5cd92dae070ca0d5", "company_id": "dhg", "reporting_year": 2024, "text": "Tốc độ tăng trưởng bình quân (CAGR) của hàng trong nước giai đoạn 2018 – 2022 đạt 6,7%, cao hơn mức tăng trưởng bình quân 4,8% của tổng doanh thu thuần."}\n{"id": "606e95bfc509ed8d", "company_id": "tra", "reporting_year": null, "text": "Dù vậy, theo Tổng cục thống kê mức tăng trưởng năm nay tuy thấp hơn mức tăng 6,68% của năm 2015 và không đạt mục tiêu tăng trưởng là 6,7% đã đề ra nhưng trong bối cảnh kinh tế thế giới không thuận lợi, trong nước gặp nhiều khó khăn thì mức trên đã là một thành công."}\n{"id": "991d5ea65a8a44c8", "company_id": "tra", "reporting_year": 2023, "text": "Tăng trưởng 11% so với năm 2021 LỢI NHUẬN SAU THUẾ | (ĐVT: Tỷ đồng) Tăng trưởng 11% so với năm 2021 Thu nhập người lao động tăng trên 13% so với 2021 Báo cáo thường niên 2022 BÁO CÁO PHÁT TRIỂN BỀN VỮNG"}\n{"id": "c385bf4d13ea40ed", "company_id": "tra", "reporting_year": null, "text": "»» Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy Tỷ lệ rác thải được tái chế chiếm 75% khối lượng rác thải của Công ty."}\n{"id": "f30d6c9cdadaf3e0", "company_id": "dhg", "reporting_year": 2024, "text": "máy Dược phẩm DHG được miễn thuế TNDN trong 04 năm đầu và giảm 50% số thuế phải nộp trong 09 năm tiếp theo (thuế suất 5%) và 2 năm còn lại thuế hằm khuyến khích đầu tư từ các doanh nghiệp, nhà nước đã có những chính sách ưu đãi đầu tư đối với khu công nghiệp, khu kinh tế."}\n{"id": "bc52592ae1a73979", "company_id": "bvh", "reporting_year": 2025, "text": "Giai đoạn tới 2025-2030 Giảm 70% lượng phát thải, tập trung vào việc loại bỏ hoàn toàn các nguồn phát thải từ năng lượng hóa • 100% hệ thống và thiết bị vận hành chạy hoàn toàn bằng năng lượng tái tạo hoặc điện sạch."}\n{"id": "ecdca2817fdc14bb", "company_id": "tra", "reporting_year": 2025, "text": "năm 2018, Hoàn thành Nghiệm thu cấp Nhà nước Dự án hỗ trợ DNKHCN, Nuôi trồng và phát triển các sản phẩm từ Đông Trùng Hạ Thảo Sa Pa Mã số Doanh thu từ các sản phẩm ĐTHT là 640 triệu chiếm tỷ lệ 20%/ tổng doanh thu nhóm SPBĐ."}\n{"id": "720efbefa9701945", "company_id": "bvh", "reporting_year": 2025, "text": "546 lít xăng được tiết kiệm trong năm 2025 đã giảm trao đổi khí CO2 thành O2 trong Chi phí và lượng xăng tiêu thụ theo năm Lượng xăng tiêu thụ (Lít) Chi phí sử dụng xăng (Triệu đồng)"}\n{"id": "ea02633f5040dd81", "company_id": "tra", "reporting_year": 2025, "text": "Tổng Công ty Đầu tư và Kinh doanh vốn Nhà nước Địa chỉ: Số 117 Trần Duy Hưng, phường Yên Hòa, Hà Nội Số cổ phần: 14.786.512 Tỷ lệ sở hữu: 35,67% SICAV-SIF - ASIA TOP Địa chỉ: 46A, avenue John F."}\n{"id": "652deffc069c8eb2", "company_id": "tra", "reporting_year": null, "text": "Tốc độ tăng trưởng của ngành cao, trung bình khoảng 14.6%/năm, vai trò và vị thế của các doanh nghiệp dược trong nước ngày càng cao;"}\n{"id": "623fca0006344f9a", "company_id": "vnm", "reporting_year": 2022, "text": "100% nhà cung cấp dịch vụ xử lý chất thải có giấy phép hành nghề."}\n{"id": "424dd7f6fb4220f3", "company_id": "bvh", "reporting_year": 2025, "text": "nếu chôn lấp không đúng cách có thể gây ô nhiễm nguồn nước và đất trong hàng chục Với mỗi viên pin, lượng thủy ngân rò rỉ có thể làm ô nhiễm tới 500 lít nước hoặc 1 mét khối đất trong 50 năm."}\n{"id": "8e336ea591852dc2", "company_id": "pan", "reporting_year": 2025, "text": "doanh thu mảng nước mắm tăng nhẹ LNG Q3/2024 tăng 41,9% svck, lũy kế tăng 37,6% svck Biên lợi nhuận gộp (%) Biên lợi nhuận cải thiện đến từ mảng nông dược (VFG), xk cá tra (ABT), snack (LAF), bánh kẹo (BBC)."}\n{"id": "125c5b70664cf1ee", "company_id": "tra", "reporting_year": null, "text": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG Tiểu ban Chính sách Phát triển Tiểu ban Nhân sự & Lương thưởng Công ty Cổ phần Dược vật tư Y tế Đắk Lắk (58%) Công ty Cổ phần Công nghệ cao Traphaco (51%) Công ty TNHH Traphaco Hưng Yên (100%) Công ty TNHH MTV Traphaco Sapa (100%) Báo cáo phát triển bền vững 2019 Tổng quan về phát triển bền vững"}\n{"id": "f34efe91c58a3d1a", "company_id": "tra", "reporting_year": null, "text": "Trong tháng 02/2011, Ngân hàng nhà nước đã tiếp tục điều chỉnh tỷ giá hối đoái từ 18.932 lên 20.693, tương đương mức tăng 9,3% giữa VND và USD, việc điều chỉnh tỷ giá tác động mạnh đến các hoạt động nhập khẩu, chi phí nguyên vật liệu đầu vào và giá máy móc thiết bị sẽ tăng."}\n{"id": "5a4578f6d804d96e", "company_id": "tra", "reporting_year": null, "text": "Nhìn lại bối cảnh trong nước và thế giới, kết quả tăng trưởng kinh tế năm 2016 ước tính tăng 6,21% là một thành công bởi kinh tế thế giới năm 2016 tiếp tục tăng trưởng ở mức thấp trong vòng 5 năm liên tiếp."}\n{"id": "15f2b4627aac7f09", "company_id": "dhg", "reporting_year": 2024, "text": "Lượng than đã tiêu thụ trong năm 2023 Hệ thống năng lượng mặt trời đã lắp đặt Tiết kiệm điện năng (*) Cắt giảm lượng khí thải CO2 (*) tương đương 373.397 kWh (*) Số liệu ước tính từ ngày 01/01/2024 - 31/05/2024 Hệ thống năng lượng mặt trời tại DHG Pharma Báo cáo thường niên 2023 GIỚI THIỆU TỔNG QUAN CHIẾN LƯỢC PHÁT TRIỂN HOẠT ĐỘNG KINH DOANH"}\n{"id": "7dcaf365fc75f298", "company_id": "ctd", "reporting_year": 2025, "text": "Tổng sản lượng điện tạo ta Mức giảm phát thải CO2 tương đương 10,78 tấn."}\n{"id": "aa42b783af8c2d31", "company_id": "dhg", "reporting_year": 2025, "text": "100% nước thải từ sản xuất và sinh hoạt được xử lý đạt chuẩn trước khi xả thải."}\n{"id": "a799c53c1ca41ae7", "company_id": "dhg", "reporting_year": 2024, "text": "Với đặc trưng nguyên liệu sản xuất phụ thuộc tới 90% vào nguồn nhập khẩu từ nước ngoài, điều này khiến các công ty trong ngành dược nói chung và DHG Pharma nói riêng chịu sự tác động mạnh mẽ từ các yếu tố bên ngoài, đặc biệt là biến động về tỷ giá."}\n{"id": "0b89929c3a86802d", "company_id": "bvh", "reporting_year": 2025, "text": "Giai đoạn tới 2030-2040 Giảm 20% lượng phát thải từ các hoạt động vận hành vào năm 2025."}\n{"id": "1cd51459250e35c0", "company_id": "pnj", "reporting_year": 2023, "text": "2021, lượng nước cấp (nước mua) và lượng nước tiêu thụ tăng mạnh so với Cụ thể, tổng lượng nước mua năm 2021 tăng 79%, tổng lượng nước tiêu thụ tăng 42% (so với năm Đó là do sự phục hồi nền kinh tế sau đợt giãn cách Covid 19, nhu cầu sản xuất tăng lên, sản lượng sản xuất đang trên đà hồi phục với mức tăng trưởng 27% trong năm vừa qua."}\n{"id": "497f36070a35a5f2", "company_id": "bvh", "reporting_year": 2025, "text": "Đáng chú ý, tỷ suất Lợi nhuận sau thuế/Vốn điều lệ (ROCC) tiếp tục vượt mức giai đoạn trước tăng vốn năm 2019 (15%), cho thấy hiệu quả sử dụng vốn ngày càng được nâng cao."}\n{"id": "34d38386ad1c87b3", "company_id": "dhg", "reporting_year": 2024, "text": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện."}\n{"id": "514d432d996634fe", "company_id": "vnm", "reporting_year": 2022, "text": "Thu mua bắp sinh khối từ hộ nông dân: 100% Đơn vị khai thác nước đúng công suất cho phép."}\n{"id": "83c5a71ed925a58e", "company_id": "dhg", "reporting_year": 2024, "text": "độ tăng trưởng bình quân (CAGR) của hàng trong nước giai đoạn 2018 – 2022 đạt 6,7%, cao hơn mức tăng trưởng bình quân 4,8% của tổng doanh thu thuần."}\n{"id": "15840ce626d9a969", "company_id": "dhg", "reporting_year": 2024, "text": "Tổng lượng nước tiêu thụ: 253.145 m3/năm."}\n{"id": "6a4dabc1e5961b92", "company_id": "pnj", "reporting_year": 2021, "text": "Bình quân (kwh/sản phẩm) Bình quân (kg/nghìn sản phẩm) Bình quân (lít/nghìn sản phẩm) BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2020 | 31"}\n{"id": "28c90fc9ae6625fa", "company_id": "ctd", "reporting_year": 2025, "text": "vững - thép xanh (tương đương 34.429 tấn) tại 17 công trường từ các nhà cung cấp có chứng nhận LEED, EPD, LCA cũng đang được đẩy mạnh sử dụng tại các công Coteccons là doanh nghiệp xây dựng tiên phong triển khai hệ thống kiểm kê các nguồn phát thải trong chuỗi hoạt động và định lượng lượng phát thải."}\n{"id": "b81d039cbf45603a", "company_id": "bvh", "reporting_year": 2025, "text": "Cường độ phát thải khí nhà kính Tấn CO2/1 tỷ đồng GDP Tấn CO2/người dân Việt Nam 6,49 Tấn CO2/người dân thế giới Cường độ phát thải khí nhà kính trung bình Gắn kết mục tiêu môi trường Lượng khí thải nhà kính tại các tòa nhà Bảo Việt (tấn CO2/1 tỷ đồng doanh thu) BÁO CÁO TÍCH HỢP 2025 ° PHÁT TRIỂN BỀN VỮNG"}\n{"id": "52389d973d5c9265", "company_id": "vnm", "reporting_year": 2022, "text": "Tỷ lệ nước tuần hoàn và tái sử dụng cho trồng Tỷ lệ tái chế phân thải bò đạt 100%."}\n{"id": "27b7575ff29307d9", "company_id": "vnm", "reporting_year": 2023, "text": "NĂNG LƯỢNG SẠCH VÀ GIÁ HỢP LÝ NƯỚC SẠCH VÀ ĐIỀU KIỆN VỆ SINH VIỆC LÀM BỀN VỮNG VÀ • 100% Đơn vị khai thác nước đúng công suất cho phép."}\n{"id": "ec5d3a191a83c2e8", "company_id": "bvh", "reporting_year": 2025, "text": "Đầu tháng 4/2025, việc Mỹ công bố áp thuế đối ứng cao đối với nhiều đối tác thương mại, trong đó mức thuế dự kiến đối với Việt Nam lên tới 46%, đã gây ra cú sốc ngắn hạn đối với thị trường tài chính toàn cầu và khiến thị trường chứng khoán trong nước điều chỉnh mạnh."}\n{"id": "5fbaf14f6e0adcdd", "company_id": "bvh", "reporting_year": 2025, "text": "Đối với lộ trình giảm phát thải hướng tới mục tiêu Net Zero, năm 2025 Bảo Việt ghi nhận mức phát thải 981,41 tấn CO2, giảm so với 996,52 tấn CO2 năm 2024, cho thấy những cải thiện nhất định trong hiệu quả sử dụng năng lượng và tối ưu vận hành."}\n{"id": "5176d3e1c44e74d4", "company_id": "dhg", "reporting_year": 2025, "text": "Điều chỉnh giới hạn tỷ lệ sở hữu của nhà đầu tư nước ngoài tại DHG Pharma từ 49% lên 100% Đạt 2 tiêu chuẩn cao PIC/s - GMP và Japan-GMP cho các dây chuyền sản xuất."}\n{"id": "a11b618c4cf29d93", "company_id": "dhg", "reporting_year": 2024, "text": "vẫn phải nhập khẩu nên giá trị sản xuất thuốc trong nước chỉ mới đạt khoảng 50% so với tổng giá trị tiền thuốc sử dụng của Bên cạnh đó, thách thức lớn nhất của ngành"}\n{"id": "f3c65e9c2af2d146", "company_id": "pnj", "reporting_year": 2023, "text": "Cường độ sử dụng nước trên 1.000 đơn vị sản phẩm tăng 3,8% so với năm 2021 Phục vụ cho việc tăng Chỉnh trang nhà xưởng Là doanh nghiệp luôn coi trọng yếu tố môi trường trong sản xuất, PNJ vẫn cố gắng xử lý tối ưu nguồn nước đầu ra, đảm bảo tỷ lệ xử lý nước trước khi thải ra môi trường ở mức 87.6% so với 2021."}\n{"id": "54005feb4da0f9d5", "company_id": "pnj", "reporting_year": 2021, "text": "nhu cầu năng lượng bình quân 5,2%/năm, việc sử dụng năng lượng tiết kiệm và hiệu quả có ý nghĩa rất quan trọng trong quá trình phát triển năng lượng và đảm bảo an ninh năng lượng quốc gia."}\n{"id": "1b2341792f537c6a", "company_id": "pow", "reporting_year": 2025, "text": "sau thuế đạt 1.130 tỷ đồng, vượt 405% so với kế hoạch Trả 7% cổ tức bằng tiền từ nguồn lợi nhuận sau thuế chưa phân phối lũy kế của Công ty vào ngày Nộp Ngân sách nhà nước năm 2025 là 207,65 tỷ đồng đạt 115% KH năm."}\n{"id": "8e42f94aa937f2ed", "company_id": "vnm", "reporting_year": 2023, "text": "125.108 m3 Lượng nước trong hoạt động sản xuất."}\n{"id": "90160994cda7bf7b", "company_id": "pan", "reporting_year": null, "text": "_ TH( 3AN GROU3 _ %ÁO CÁO TH\xad ́NG NI\x92N \\u0015\\u0013\\u00146 ĐÁNH GIÁ CỦA HỘI ĐỒNG QUẢN TRỊ VỀ HOẠT ĐỘNG CỦA CÔNG TY SUSTAINABILITY REPORT | THE PAN GROUP OVERVIEW Food safety issues have also caused concerns for both the public and government and monitoring mechanisms have become stricter in response to the increasing number and severity of food safety violations."}\n{"id": "a2a3a106e156efee", "company_id": "pnj", "reporting_year": 2023, "text": "- Giảm phát thải ra môi trường 7800m3 nước thải /năm."}\n{"id": "61af863cb59bec69", "company_id": "vnm", "reporting_year": 2022, "text": "85% Lượng nước thải chăn nuôi xử lý đạt chuẩn được tuần hoàn và tái sử dụng cho trồng trọt."}\n{"id": "df2dd928f2632597", "company_id": "pnj", "reporting_year": 2021, "text": "Ngoài việc tiêu thụ năng lượng điện phục vụ cho các hoạt động của mình, PNJP còn có một máy phát điện dự phòng với công suất 870KVA để sử dụng trong trường hợp mất điện, nhiên liệu dùng cho máy phát điện là 120 lít dầu DO dùng cho 1 giờ mất điện."}\n{"id": "753db5fc285fbc10", "company_id": "bvh", "reporting_year": 2025, "text": "GDP cả nước đạt mức tăng 8,02% so với năm trước, mức tăng cao thứ hai trong giai đoạn 2011–2025, chỉ thấp hơn năm 2022."}\n{"id": "7b233c31ddac18d6", "company_id": "ctd", "reporting_year": 2025, "text": "Đây là loại sản phẩm đồng phục thân thiện với môi trường được làm từ vải nano (dệt từ 100% chai nhựa tái chế) và vải cafe (dệt từ chai nhựa tái chế và bã cafe)."}\n{"id": "6d85557349a3e68b", "company_id": "dhg", "reporting_year": 2024, "text": "Con số này vừa thể hiện cơ hội, tiềm năng nhưng cũng là thách thức không nhỏ cho các doanh nghiệp dược trong nước để phấn đấu đưa giá trị thuốc sản xuất trong nước chiếm 70% thị phần theo chủ trương của Chính Phủ."}\n{"id": "1ed42a8889b63e5e", "company_id": "dhg", "reporting_year": 2025, "text": "Tổng Công ty Đầu tư và Kinh doanh vốn Nhà nước (“SCIC”), sở hữu 43,31% vốn cổ phần của Công ty."}\n{"id": "25a034a57e7e9f6d", "company_id": "bvh", "reporting_year": 2025, "text": "chuyển đổi số được đẩy mạnh toàn diện, hướng tới nâng tỷ trọng kinh tế số lên khoảng 22 - 25% GDP, tạo động lực tăng trưởng mới và bền vững cho nền kinh tế trong trung và dài hạn."}\n{"id": "7e519ee1ab65b89c", "company_id": "bvh", "reporting_year": 2025, "text": "Lượng nước thải tại các tòa nhà năm 2025 (m3) 8 Lê Thái Tổ 71 Ngô Sĩ Liên 7 Lý Thường Kiệt 233 Đồng Khởi"}\n{"id": "cd85808df2b47994", "company_id": "vnm", "reporting_year": 2023, "text": "Toàn tập đoàn có 9.877 nhân sự tại Chúng tôi không bị giới hạn tỷ lệ sở hữu Cơ cấu cổ đông được duy trì đa dạng với tỷ lệ nhà đầu tư (“NĐT”) nước ngoài/trong nước là 54%/46% và NĐT tổ chức/cá nhân là 92%/8% (tại Chúng tôi hiện có 250 SKUs trong danh mục sản phẩm với 04 ngành hàng chính gồm Sữa nước, Sữa chua, Sữa đặc và Sữa công thức."}\n{"id": "b82cbb12e632d3dc", "company_id": "vnm", "reporting_year": 2022, "text": "trong hoạt động tại hệ thống trang trại góp phần tích cực vào mục tiêu giảm phát thải khí nhà kính với khả năng làm giảm lên đến 62.000 tấn CO2/năm tương đương với việc trồng 3,4 Thực hành nông nghiệp tái tạo CÔNG TY CỔ PHẦN SỮA VIỆT NAM CHUYỂN ĐỔI - THÚC ĐẨY TĂNG TRƯỞNG Báo cáo thường niên 2022"}\n{"id": "fc39cc345f8345b8", "company_id": "bvh", "reporting_year": 2025, "text": "Tỷ lệ thôi việc năm 2025 của Bảo Việt tiếp tục nằm trong ngưỡng ổn định theo thông lệ quản trị nhân sự (4-6%), thấp hơn đáng kể so với tỷ lệ nghỉ việc tự nguyện bình quân của doanh nghiệp trong nước năm 2025 (9,6%, theo khảo sát Talentnet-Mercer)."}\n{"id": "43e786abfdfadb1e", "company_id": "ctd", "reporting_year": 2025, "text": "Theo ước tính, ngành này tiêu thụ khoảng 40% tổng năng lượng toàn cầu, chịu trách nhiệm cho hơn 30% tổng lượng phát thải khí nhà kính và sử dụng một lượng"}\n{"id": "ebf49553973f71fd", "company_id": "bvh", "reporting_year": 2025, "text": "Sử dụng vốn hiệu quả thông qua việc đầu tư vào các tài sản an toàn (tỷ trọng trái phiếu Chính phủ và tiền gửi luôn chiếm trên 90% tổng danh mục đầu tư);"}\n{"id": "139eb880d3de9124", "company_id": "vnm", "reporting_year": 2022, "text": "Trồng cây cung cấp 37.100 tấn thức ăn thô xanh cho bò sữa và luân canh cây họ đậu, phủ xanh 1.182 ha diện tích đất đồng cỏ."}\n{"id": "e6ae7b647d585771", "company_id": "bvh", "reporting_year": 2025, "text": "Ở chiều ngược lại, nền kinh tế trong nước duy trì tăng trưởng tích cực, với GDP ước đạt 8,02%, các định hướng phát triển rõ ràng, mặt bằng lãi suất trong nước trong những tháng đầu năm tiếp tục được duy trì ở mức thấp"}\n{"id": "bee9d16171861ba5", "company_id": "bvh", "reporting_year": 2025, "text": "Lắp đặt hệ thống tái chế rác thải hoặc tái sử dụng nhằm giảm tác động lên • 50% tòa nhà lắp đặt hệ thống năng lượng tái tạo và giảm thiểu sử dụng năng lượng không tái tạo."}\n{"id": "36b0d339f41dceb8", "company_id": "dhg", "reporting_year": 2024, "text": "Tổng lưu lượng nước thải phát sinh trong năm 2022 khoảng 22.909 m3/năm."}\n{"id": "ebf98e6bd3bc6405", "company_id": "vnm", "reporting_year": 2022, "text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu: 86,8% Trụ sở chính, 11 Nhà Máy, 13 Trang trại được lắp đặt Hệ thống năng lượng mặt trời."}\n{"id": "6b9243d45d24de6d", "company_id": "pnj", "reporting_year": 2023, "text": "NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU Năm 2021 là một năm chứng kiến những diễn biến phức tạp và kéo dài của dịch Covid-19, thời gian đóng cửa của PNJ chiếm khoảng 22% thời gian trong năm;"}\n{"id": "d5dced6346811c4c", "company_id": "dhg", "reporting_year": 2024, "text": "Hàng trong nước đóng vai trò cốt lõi trong việc tăng trưởng của doanh thu thuần, với giá trị đóng góp 562 tỷ Doanh thu thuần hàng trong nước năm 2022 đạt 3.910 tỷ đồng, tăng 16,8% so với cùng kỳ, đây là mức tăng trưởng cùng kỳ cao nhất trong 5 năm gần đây."}\n{"id": "07a9eaef4ce5d77e", "company_id": "vnm", "reporting_year": 2023, "text": "Vinamilk là công ty đầu tiên và duy nhất của Châu Á sở hữu giải thưởng Purity Award cho sữa bột Sữa tươi Vinamilk Green Farm, Vinamilk 100% Organic, Vfresh Nước ép Lựu Táo, Vfresh Trà Atiso không đường và sữa đặc Ngôi Sao Phương Nam."}\n{"id": "a8edb60c54a6e09b", "company_id": "dhg", "reporting_year": 2024, "text": "Sau khi chất thải được chuyển đi, nhân viên vệ sinh công nghiệp quét dọn, khử trùng bằng dung dịch javel 0,4%."}\n{"id": "2f66f3e51fd1ce15", "company_id": "pnj", "reporting_year": null, "text": "Lưu lượng nước thải sản xuất phát sinh khoảng 50~60 m3/ Thành phần nước thải này chứa chủ yếu là các kim loại nặng từ quá trình phân tách kim loại để thu hồi các kim loại quý, cặn vô cơ, bụi thạch cao..."}\n{"id": "c46b97cef8117154", "company_id": "pnj", "reporting_year": 2023, "text": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3."}\n{"id": "1983688bb144c335", "company_id": "pnj", "reporting_year": 2023, "text": "Khả năng sinh lời vượt trội mang lại lợi ích cho cổ đông Tỷ lệ ROE tăng từ mức 18,3% (2021) lên mức 25% cho thấy công ty sử dụng vốn hợp lý và hiệu quả."}\n{"id": "999aac0d083169fb", "company_id": "vnm", "reporting_year": 2022, "text": "Purity Award, sản phẩm sữa tươi tiệt trùng Vinamilk Green Farm và Vinamilk 100% Organic cũng là các sản phẩm sữa tươi đầu tiên trên thế giới đạt được chứng nhận về Clean Label cho các thương hiệu không chỉ có sự minh bạch về sản phẩm"}\n{"id": "10ed2b98b6019317", "company_id": "pnj", "reporting_year": null, "text": "HÀNH TRÌNH PHÁT TRIỂN BỀN VỮNG ĐIỂN HÌNH CHÂU Á GIẢI THƯỞNG VÀ CON SỐ TIÊU BIỂU Ngành kim hoàn châu Á LÃI RÒNG:1.190 TỶ ĐỒNG BIÊN LỢI NHUẬN: 20,4% MẠNG LƯỚI BÁN LẺ: 346 CỬA HÀNG GIÁ TRỊ THƯƠNG HIỆU: MÁI ẤM NIỀM TIN: 300 CĂN HOẠT ĐỘNG CSR:11,7TỶ ĐỒNG"}\n{"id": "7bfb7254afb46d03", "company_id": "vnm", "reporting_year": 2023, "text": "Thị trường nước ngoài Doanh thu thuần của thị trường nước ngoài đạt 9.752 tỷ đồng, tăng 5,4% so với cùng kỳ, trong đó, Các chi nhánh nước ngoài đóng góp 5.039 tỷ đồng và hoạt động Xuất khẩu đóng góp 4.713 tỷ đồng, tăng lần lượt 6,5% và 4,4% so với Lợi nhuận gộp từ Thị trường nước ngoài đạt 3.651 tỷ đồng, tăng ấn tượng 9,3% so với cùng kỳ."}\n{"id": "921b077d27d528bf", "company_id": "vnm", "reporting_year": 2023, "text": "BÁO CÁO KẾT QUẢ HOẠT ĐỘNG SẢN XUẤT KINH DOANH NĂM 2023 Hệ thống trang trại của Vinamilk tiếp tục thực hành sản xuất nông nghiệp bền vững với 100% trang trại đạt chuẩn GlobalG.A.P và điểm nhấn chủ đạo là hệ thống các trang trại sinh thái Vinamilk Green Farm và Trang trại Organic theo tiêu chuẩn châu Âu."}\n{"id": "f0dddfad2f9461c6", "company_id": "dhg", "reporting_year": 2025, "text": "nhân mà thuốc trong nước chỉ chiếm gần 50% thị phần cả nước."}\n{"id": "5da9259d021e6639", "company_id": "dhg", "reporting_year": 2024, "text": "Chính phủ Việt Nam đặt ra một số mục tiêu trọng tâm GDP bình quân đầu người đạt (USD) Tăng trưởng GDP khoảng (%) Động lực chính cho tăng trưởng là (1) hồi phục nhu cầu trong nước và thế giới, (2) đẩy mạnh đầu tư công, (3) hồi phục của thị trường bất động sản và (4) lĩnh vực đầu tư tư nhân."}\n{"id": "9e1e32843cacdcab", "company_id": "bvh", "reporting_year": 2025, "text": "Riêng trong năm 2025, dù đối mặt với nhiều biến động và bất ổn của nền kinh tế, Tập đoàn vẫn thực hiện chi trả cổ tức ở mức 10,551%, tương đương hơn 783 tỷ đồng, qua đó tiếp tục khẳng định cam kết bảo đảm quyền lợi cổ đông cũng như hiệu quả hoạt động ổn định và bền vững của Tập đoàn."}\n{"id": "9abb770f0de72be6", "company_id": "bvh", "reporting_year": 2025, "text": "tổng dự phòng nghiệp vụ bảo hiểm tăng 8,8%, cho thấy khả năng đảm bảo nghĩa vụ chi trả trong tương lai của các doanh nghiệp được duy trì Bên cạnh đó, tổng nguồn vốn chủ sở hữu của các doanh nghiệp bảo hiểm tăng 3,8%, phản ánh nỗ lực tăng cường năng lực tài chính và đáp ứng các yêu cầu quản lý ngày càng chặt chẽ của cơ quan quản lý nhà nước."}\n{"id": "18457bf2b1c96743", "company_id": "bvh", "reporting_year": 2025, "text": "Hợp đồng thu gom rác • Cung cấp công cụ thu • Thu gom chất thải phát sinh trong hoạt động sản • Số lượng thu gom: 150kg/ • Cung cấp phương tiện vận chuyển và công cụ lưu trữ."}\n{"id": "7ae3ded84e6645ec", "company_id": "bvh", "reporting_year": 2025, "text": "bảo hiểm đạt trên 12.300 tỷ đồng, tăng 8,4%, phản ánh sự tăng trưởng ổn định và bền vững trong hoạt động kinh doanh."}\n{"id": "53f64ffeb4c03d32", "company_id": "vnm", "reporting_year": 2022, "text": "Các chi nhánh nước ngoài là đầu tàu tăng trưởng cho hoạt đông kinh doanh quốc tế của Vinamilk doanh thu ghi nhận mức tăng trưởng trên 30% trong năm 2022 nhờ đa dạng hóa kênh phân phối để bù đắp cho kênh chủ lực trường học vẫn chưa phục hồi bằng mức trước Ngoài ra, Driftwood duy trì được việc đưa một số sản phẩm của Vinamilk vào thị trường Mỹ."}\n{"id": "11bff39daa1d0a60", "company_id": "vnm", "reporting_year": 2022, "text": "100% Nước thải trong hoạt động Sản xuất và Chăn nuôi được xử lý đạt chuẩn."}\n{"id": "a345d2e5749efda5", "company_id": "ctd", "reporting_year": 2025, "text": "ban đầu, việc sử dụng bê tông khoáng hóa CO2 (bê tông Fico) với khối lượng 132.603 tấn tiết giảm được 1.936 tấn CO2 ra môi trường trong năm tài chính vừa qua."}\n{"id": "646353a5f7d99d72", "company_id": "pnj", "reporting_year": null, "text": "sách ưu tiên cho nhà cung cấp trong nước, kể cả mức giá cao hơn (mức chấp nhận được) đối với nguồn hàng có chất NĂNG LỰC QUẢN TRỊ CHUỖI CUNG ỨNG THU HÚT KHÁCH HÀNG MỚI VÀ CHĂM SÓC KHÁCH HÀNG Thường xuyên phân tích hàng vi khách hàng hiện hữu, chăm sóc và gia tăng độ tương tác, đẩy doanh thu tập khách hàng VIP lên 50,1% trên tổng doanh thu bán lẻ."}\n{"id": "1fd039736163702d", "company_id": "dhg", "reporting_year": 2025, "text": "(Nguồn: IQVIA Quý 4/2024 – số liệu dựa trên ước tính doanh số bán của IQVIA) Công ty nước ngoài A Công ty nước ngoài B Công ty trong nước C Công ty trong nước D Tổng thể, mặc dù kênh doanh thu chủ lực (Pharmacy) sụt giảm trong năm 2024, nhưng được kỳ vọng sẽ phục hồi với mục tiêu tăng trưởng 6% trong năm 2025."}\n{"id": "14c0acc8a4a7b20e", "company_id": "dpr", "reporting_year": 2024, "text": "Các sản phẩm đã và đang được xuất khẩu sang các nước như Trung Quốc, Singapo, Hàn Quốc, Đài Loan, Đức, Malaisia, Nhật, Pháp, Hà Lan, Tây Ban Nha, Anh, Tình hình kinh tế vĩ mô của Việt Nam tương đối ổn định, tốc độ tăng trưởng GDP tăng 5,05% Sản lượng xuất khẩu cao su khả quan hơn bởi ngành săm lốp tại Trung Quốc phục hồi trở lại."}\n{"id": "3fed2faa180e1937", "company_id": "pnj", "reporting_year": 2023, "text": "góp phần hoàn thành cam kết giảm thiểu khí nhà kính theo lộ trình mục tiêu quốc gia và đặt mục tiêu giảm thải khí nhà kính cho 02 nhà máy là 2% (đơn vị: tấn phát thải carbon trên 1.000 đơn vị sản phẩm trong 1 năm) - bám sát mục tiêu giảm phát ròng về “0” của Chính phủ Việt Nam."}\n{"id": "a95f4adb37189ab0", "company_id": "ctd", "reporting_year": 2025, "text": "Coteccons Group cũng vừa chào đón một thành viên mới ở mảng MEP là UG Vietnam khi hoàn thành mua 100% doanh nghiệp này để sáp nhập vào hệ sinh thái."}\n{"id": "2d1b0b7ae3d3450d", "company_id": "dhg", "reporting_year": 2024, "text": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3."}\n{"id": "ca96c83290aa0876", "company_id": "bvh", "reporting_year": 2025, "text": "Mức nộp Ngân sách Nhà nước năm 2022 vẫn đạt mức cao nhất do khoản chi trả cổ tức kỷ lục ở mức 30,261% với phần cổ tức chi trả cho cổ đông Bộ Tài chính là 1.460 tỷ đồng."}\n{"id": "0e706e79cf6d7000", "company_id": "dhg", "reporting_year": 2023, "text": "Điều chỉnh giới hạn tỷ lệ sở hữu của nhà đầu tư nước ngoài tại DHG Pharma từ 49% lên 100% vốn điều lệ."}\n{"id": "2aa0d2715bda9128", "company_id": "bvh", "reporting_year": 2025, "text": "mỗi viên pin, lượng thủy ngân rò rỉ có thể làm ô nhiễm tới 500 lít nước hoặc 1 mét khối đất trong 50 năm."}\n{"id": "9a9a5f5612c171c6", "company_id": "pnj", "reporting_year": 2023, "text": "Tỷ lệ nước thải đã xử lý / nước cấp (%) vào năm 2020 là 73.74% so với 83.94% của năm 2021 với lý do thất thoát ở một số đường ống phân phối nước và chưa tối ưu lượng nước dùng cho tưới cây để duy trì mảng xanh công nghiệp."}\n{"id": "8b4614460f72f65c", "company_id": "pow", "reporting_year": 2025, "text": "Công nghiệp - Năng lượng Quốc gia Việt Nam sở hữu 100% vốn điều lệ) phải trả cho Tập đoàn Công nghiệp - Năng lượng Quốc gia Việt Nam với số tiền là 728.174.152.403 VND (tại ngày 31 tháng 12 năm 2024: 1.028.174.152.403 VND), theo hướng dẫn của Công văn số 13683/BTC-TCDN ngày 06 tháng 11 năm 2020 của Cục Tài chính Doanh nghiệp - Bộ Tài chính."}\n{"id": "5d74fc3fa628f4a0", "company_id": "dhg", "reporting_year": 2024, "text": "100% nước thải từ hoạt động sản xuất và sinh hoạt được xử lý đạt quy chuẩn trước khi xả thải vào nguồn tiếp nhận."}\n{"id": "1513369675a9f076", "company_id": "pnj", "reporting_year": 2023, "text": "Số hộ gia đình khó khăn được PNJ hỗ trợ tỷ đồng (+11.6% so với 2020) Tổng sản lượng sản phẩm sản phẩm (+20% so với 2020) cửa hàng trên toàn quốc Đóng góp ngân sách nhà nước Hỗ trợ cộng đồng vượt bão COVID-19"}\n{"id": "fb311be37160e5bc", "company_id": "pan", "reporting_year": 2025, "text": "BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 In 2020, Vietnam committed to reducing greenhouse gas (GHG) emissions by 15.8% by 2030, and up to 43.5% with international support."}\n{"id": "d3519dd0a0cc1dd7", "company_id": "pow", "reporting_year": 2025, "text": "trường điện thực tế trong 11 tháng đầu năm 2025 rất thấp, cụ thể: giá điện năng thị trường (SMP) trung bình là 847,46 (đồng/kWh), giá công suất thị trường (CAN) là 47,81 (đồng/kWh), giá thị trường điện toàn phần (FMP) trung bình đạt 895,27 (đồng/kWh)."}\n{"id": "8a12299d98291537", "company_id": "pnj", "reporting_year": 2023, "text": "Trong bối cảnh năng suất tăng hơn 60%, áp lực đảm bảo chỉ tiêu cường độ tiêu hao năng lượng là Dù chưa được như kỳ vọng nhưng mức giảm tiêu thụ gas 1,4% so với năm 2021 là kết quả đáng ghi nhận."}\n{"id": "b462cf145b84bc0b", "company_id": "pnj", "reporting_year": 2023, "text": "QCVN 40:2011/BTNMT CỘT B NƯỚC TUẦN HOÀN PHA LOẢNG ĐƯỜNG HÓA CHẤT/ DƯỠNG CHẤT ĐƯỜNG DỰ PHÒNG/ CHẢY TRÀN CHỨA NƯỚC PK SAU ÉP BÙN ĐƯỜNG VỀ HỐ GA THU NƯỚC CHỨA NT SÚC XẢ XI MẠ The process flow diagram for wastewater treatment at PNJ company (Q=50m3/day and night) Moving forward, PNJP is committed to achieving 40:2011/BTNMT) in 2023."}\n{"id": "a29b6a8bd80b1cb5", "company_id": "bvh", "reporting_year": 2025, "text": "Giai đoạn tới 2025-2030: Giảm 50% lượng phát thải thông qua việc tối ưu hóa sử dụng năng lượng và phương tiện di chuyển vào năm 2030."}\n{"id": "f15d875fd9b8cd3a", "company_id": "dhg", "reporting_year": 2025, "text": "422 tỷ đồng vào ngân sách Nhà nước và dự kiến chi trả cổ tức năm 2025 ở mức 100% (tương đương 10.000 đồng/cổ phiếu) — mức cao nhất trong số các doanh nghiệp Dược Chiến lược R&D định hướng thị trường Mọi sản phẩm của DHG Pharma đều bắt nguồn từ nhu cầu điều trị thực tiễn của người bệnh."}\n{"id": "69c1789339f33228", "company_id": "bvh", "reporting_year": 2025, "text": "Lợi suất trái phiếu Chính phủ kỳ hạn 10 năm dao động quanh 4,1 - 4,3%, phản ánh mặt bằng lãi suất trong nước và điều kiện thanh khoản của hệ thống tài chính."}\n{"id": "d63df9b323f300f4", "company_id": "bvh", "reporting_year": 2025, "text": "hiểm Bảo Việt tiếp tục duy trì đà tăng trưởng tích cực, với tổng doanh thu đạt 12.920 tỷ đồng, tăng 8,3% so với cùng kỳ và vượt kế hoạch đề Doanh thu phí bảo hiểm đạt trên 12.300 tỷ đồng, tăng 8,4%, phản ánh sự tăng trưởng ổn định và bền vững trong hoạt động kinh doanh."}\n{"id": "7227d2158d4d5371", "company_id": "bvh", "reporting_year": 2025, "text": "Phát động chiến dịch “Tiết kiệm một giọt – Góp phần cho hành tinh xanh” nhằm thay đổi hành vi sử dụng nước trong Biện pháp xử lý kỹ thuật Giải pháp cộng đồng & Truyền thông nội bộ 29.863 m3 nước thải được thu gom, xử lý thay vì thải ra môi trường tương đương giảm phát thải"}\n{"id": "31c5e10fca0f0f0d", "company_id": "dhg", "reporting_year": 2024, "text": "Hội thảo “Chăm sóc sức khỏe phòng bệnh ở người lớn tuổi bằng tinh hoa Việt Nhật” đồng hành bởi nhãn hàng NattoEnzym với sự tham gia của hơn 7.000 khách hàng đến từ các nhà thuốc trên khắp cả nước."}\n{"id": "7f6838ba63af73c5", "company_id": "bvh", "reporting_year": 2025, "text": "Lan tỏa ý thức bảo vệ môi trường 2: CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG Cường độ phát thải khí nhà kính tại Bảo Việt (Tấn CO2/1 tỷ đồng doanh thu)"}\n{"id": "d75f5077e36f39e8", "company_id": "pow", "reporting_year": 2025, "text": "Nộp Ngân sách Nhà nước: đồng, đạt 172% kế hoạch."}\n{"id": "4c5d92717949617a", "company_id": "ctd", "reporting_year": 2025, "text": "Theo báo cáo của Bộ Xây dựng, tốc độ tăng trưởng của ngành xây dựng trong nước đạt 7,34% trong 6 tháng của hoạt động kinh doanh BĐS tăng 2,45% so với cùng kỳ năm ngoái."}\n{"id": "bf4664b800389e94", "company_id": "bvh", "reporting_year": 2025, "text": "Biện pháp xử lý kỹ thuật Giải pháp cộng đồng & Truyền thông nội bộ 41,8 tấn rác thải sinh hoạt được thu gom, tái chế thay vì thải ra môi trường tương đương giảm phát thải"}\n{"id": "2e2b8ddd78b227ba", "company_id": "pow", "reporting_year": 2025, "text": "Các khó khăn thách thức năm 2025 Tỷ trọng các nhà máy điện năng lượng tái tạo vẫn được ưu tiên (thực tế 10 tháng đầu năm chiếm 12% sản lượng điện hệ thống) dẫn tới các nhà máy nhiệt điện phải giảm phát trong các chu kỳ cao điểm ngày, làm sụt giảm sản lượng và doanh thu của các nhà máy."}\n{"id": "7bae1fe7cdef2a0c", "company_id": "bvh", "reporting_year": 2025, "text": "Lượng phát thải khí nhà kính từ hoạt động này ghi nhận 58,72 tấn CO2, tăng so với mức 45,95 tấn CO2 năm 2024, chủ yếu do nhu cầu mở rộng hoạt động kinh doanh và gia tăng tần suất di chuyển phục vụ công tác điều hành và phát triển thị trường."}\n{"id": "aebfd723671b74e8", "company_id": "pow", "reporting_year": 2025, "text": "Đối với Nhà máy điện Nhơn Trạch 2, Công ty Cổ phần Điện lực Dầu khí Nhơn Trạch 2 (NT2 - Công ty con của Tổng công ty) có nghĩa vụ phải nộp thuế thu nhập doanh nghiệp theo tỷ lệ 10% tính trên thu nhập chịu thuế từ hoạt động sản xuất và kinh doanh điện năng trong 15 năm kể từ năm đầu tiên có doanh thu và thuế suất hiện hành cho những năm tiếp theo."}\n{"id": "5167e31e2006d0b5", "company_id": "dhg", "reporting_year": 2025, "text": "HỆ THỐNG KIỂM SOÁT NỘI BỘ NỀN TẢNG VỮNG CHẮC CHO SỰ PHÁT TRIỂN BỀN VỮNG Tốt: Thực hiện 100% thông lệ Chưa đáp ứng: Chưa đáp ứng theo thông lệ Chưa đầy đủ: đã thực hiện được một phần nhưng chưa đầy đủ theo thông lệ Báo cáo thường niên 2024"}\n{"id": "d4094d82d43e0614", "company_id": "ctd", "reporting_year": 2025, "text": "Với mục tiêu đạt tỷ lệ tăng trưởng kép hàng năm (CAGR) 20-30% trong 5 năm tới, hướng tới phát triển bền vững trên vai trò industry leader ngành xây dựng tại Việt Nam và hiện diện vững chắc trên thị trường nước ngoài, Coteccons xác định 6 ưu tiên chiến lược để nhất quán CÂU CHUYỆN SAU NHỮNG CON SỐ BÁO CÁO THƯỜNG NIÊN NĂM TÀI CHÍNH 2024"}\n{"id": "d65b623ec6a4814e", "company_id": "pnj", "reporting_year": 2021, "text": "Nước dùng cho việc kiểm tra hệ thống PCCC nội bộ và diễn tập PCCC Giảm tiêu hao năng lượng Chi phí đầu tư vào các sáng kiến tiết kiệm Chi phí tiết kiệm được Tổng lượng nước tái chế/tái sử dụng (m3) Tỷ lệ nước tái chế/tái sử dụng (%) Tổng lượng nước thải theo chất lượng và địa điểm BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2020 | 33"}\n{"id": "7d48037be0905873", "company_id": "vnm", "reporting_year": 2023, "text": "lượng sữa tươi nguyên liệu thu mua từ các hộ nông dân hợp tác đạt 240.000 tấn/năm trên phạm vi cả nước."}\n{"id": "3d2e92faba94ee99", "company_id": "ctd", "reporting_year": 2025, "text": "CHÂN KIỀNG CHO COTECCONS Trong năm qua, Coteccons ghi nhận nhóm sản phẩm xây dựng công nghiệp - nhà máy đã đóng góp đến 50% trong tổng doanh thu của toàn Phần lớn khách hàng của mảng này là doanh nghiệp có vốn đầu tư nước ngoài - FDI."}\n{"id": "61070f16e7d05adf", "company_id": "dhg", "reporting_year": 2024, "text": "Việc triển khai dự án năng lượng mặt trời, ước tính lũy kế trong 20 năm sẽ giúp DHG Pharma giảm chi phí tiền điện là 57 tỷ đồng và cắt giảm lượng khí thải CO2 khoảng 53 nghìn tấn."}\n{"id": "d08482796aa8de43", "company_id": "vnm", "reporting_year": 2022, "text": "Không có khiếu nại của cộng đồng xung quanh về Không có khiếu nại về bất bình đẳng Không có khiếu nại về phân biệt đối xử Không lao động cưỡng bức, không lao động trẻ em 100% Nước thải hoạt động sản xuất được xử lý trước khi thải ra môi trường."}\n{"id": "025e1ad6d25b33e3", "company_id": "vnm", "reporting_year": 2023, "text": "Năm 2023 cũng đánh dấu sự ra mắt của các sản phẩm mới hay các sản phẩm cải tiến trong diện mạo mới được người tiêu dùng đón nhận Ngoài Sữa tươi Vinamilk 100%, Sữa tươi Green Farm và Sữa dinh dưỡng trong bịch Fino theo định hướng thương hiệu Masterbrand mới, còn có Sữa chua ăn bộ đôi dưỡng nhan bổ sung collagen Ít đường Nha Đam và Lựu đỏ;"}\n{"id": "940669fbdb2d9e49", "company_id": "bvh", "reporting_year": 2025, "text": "Tuy nhiên, tỷ lệ 27,7% vật liệu chưa thể tái chế vẫn là dư địa cần cải thiện, đặc biệt đối với các nhóm vật liệu khó xử lý như nhựa hỗn hợp và thiết bị điện tử."}\n{"id": "48e2de3d226bedd1", "company_id": "bvh", "reporting_year": 2025, "text": "Cường độ sử dụng năng lượng Cường độ sử dụng năng lượng trung bình tại Việt Nam KWH/1 TỶ ĐỒNG DOANH THU NĂM 2025 KWH/1 TỶ ĐỒNG DOANH THU 134.580 kWh điện năng được giảm tiêu thụ trong năm 2025 tương đương giảm phát thải cây xanh trao đổi khí CO2 thành O2 TÚI RÁC ĐƯỢC TÁI CHẾ"}\n{"id": "2acdf56b4cf2c9c4", "company_id": "ctd", "reporting_year": 2025, "text": "Đặc biệt, tại các công trường này, 50% hệ thống chiếu sáng sử dụng đèn năng lượng mặt trời."}\n{"id": "1a9fb56a0423c593", "company_id": "vnm", "reporting_year": 2023, "text": "Công ty đã nhận được giấy chứng nhận đầu tư ra nước ngoài số 663/BKHĐT-ĐTRNN do Bộ Kế hoạch và Đầu tư cấp, đầu tư vào Công ty Driftwood Dairy Holding Corporation tại Bang California, Tại ngày 31 tháng 12 năm 2013, Công ty đã hoàn tất thủ tục chuyển tiền đầu tư 7 triệu USD và nắm giữ 70% vốn chủ sở hữu tại Driftwood Dairy Holding Corporation."}\n{"id": "1d19e242d14134a1", "company_id": "vnm", "reporting_year": 2023, "text": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất)."}\n{"id": "c89c1cce319ca71e", "company_id": "pnj", "reporting_year": 2023, "text": "Khi cả nước gồng mình chống dịch, PNJ là một trong số ít doanh nghiệp vượt bão thành công khi đạt gần như toàn bộ kế hoạch đã đặt ra, chạm mức tăng trưởng 10% và gia Thành quả ấy có được, từ sự nỗ lực không mệt mỏi, sức sáng tạo vô giới hạn, biết tận dụng “cơ trong nguy” của cả hệ thống."}\n{"id": "f5f6919274494174", "company_id": "ctd", "reporting_year": 2025, "text": "Bên cạnh đó, trong tháng 02/2024, Coteccons đã hoàn tất mua lại 100% đối với 2 doanh nghiệp là Sinh Nam (lĩnh vực sản phẩm chính là nhôm - kính facade) và ty này nhằm hoàn thiện chuỗi cung ứng để Coteccons thuận tiện mở rộng phân khúc sản phẩm - dịch vụ, tiến ra thị trường nước ngoài và cải thiện biên lợi nhuận."}\n',
    'jobs_c.json': '[\n {"tag": "q3-32b-awq", "tp": 2, "max_len": 4096, "env": {"EXTRA_INPUT": "candidates.jsonl"}, "cands": [{"repo": "Qwen/Qwen3-32B-AWQ"}]}\n]\n',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: python -u multirun.py jobs_c.json ag1_worker.py', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u multirun.py jobs_c.json ag1_worker.py')


In [ ]:
print('[cell] run 2: ls -la out; cat out/*_summary.json', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('ls -la out; cat out/*_summary.json')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)